# Clean Two-Path Breast Cancer AI Notebook

هذه النوتبوك هي النسخة النظيفة الكاملة للمشروع.

لا تستدعي أي سكريبتات خارجية من المشروع. كل الكود موجود داخل النوتبوك نفسها.

الملفات الخارجية الوحيدة المستخدمة هي ملفات الداتا الخام:

- BCSC Risk Estimation Dataset
- METABRIC cBioPortal clinical + gene-expression files

## Path 1

Before diagnosis: BCSC future breast cancer risk.

## Path 2

After diagnosis: METABRIC molecular subtype + 5-year recurrence/survival + gene explainability + doctor-support recommendations.

In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "joblib": "joblib",
    "docx": "python-docx",
    "xgboost": "xgboost",
    "lightgbm": "lightgbm",
    "imblearn": "imbalanced-learn",
    "lifelines": "lifelines",
}
missing = [pip_name for module, pip_name in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
print("Dependencies ready")


In [ ]:
from __future__ import annotations

import json
import os
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from docx import Document
from docx.enum.table import WD_ALIGN_VERTICAL, WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
PROJECT_DIR = Path.cwd()
BCSC_DATA_DIR = PROJECT_DIR / "external_datasets" / "BCSC"
METABRIC_DATA_DIR = PROJECT_DIR / "external_datasets" / "METABRIC" / "brca_metabric_direct"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "clean_two_path_complete"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "deliverables"

for path in [OUTPUT_DIR, FIGURE_DIR, MODEL_DIR, REPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Output:", OUTPUT_DIR)

## Helper Functions

In [ ]:
def save_json(obj: dict, path: Path) -> None:
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def save_fig(path: Path) -> None:
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()


def choose_threshold(y_true: pd.Series, scores: np.ndarray, metric: str = "balanced_accuracy") -> tuple[float, float]:
    best_threshold = 0.5
    best_score = -1.0
    for threshold in np.unique(np.quantile(scores, np.linspace(0.01, 0.99, 199))):
        pred = (scores >= threshold).astype(int)
        if metric == "f1":
            score = f1_score(y_true, pred, zero_division=0)
        else:
            score = balanced_accuracy_score(y_true, pred)
        if score > best_score:
            best_threshold = float(threshold)
            best_score = float(score)
    return best_threshold, best_score


def binary_metrics(y_true: pd.Series, scores: np.ndarray, threshold: float) -> dict:
    pred = (scores >= threshold).astype(int)
    return {
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "average_precision": float(average_precision_score(y_true, scores)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "threshold": float(threshold),
    }


def horizon_label(events: pd.Series, months: pd.Series, horizon: float = 60.0) -> pd.Series:
    y = pd.Series(index=events.index, dtype=float)
    y[(events.eq(1)) & (months <= horizon)] = 1.0
    y[(months >= horizon) & ~((events.eq(1)) & (months <= horizon))] = 0.0
    return y

## Path 1: BCSC Future Breast Cancer Risk

In [ ]:
BCSC_COLUMNS = [
    "menopaus", "agegrp", "density", "race", "hispanic", "bmi",
    "agefirst", "nrelbc", "brstproc", "lastmamm", "surgmeno", "hrt",
    "invasive", "cancer", "training", "count",
]
BCSC_FEATURES = [
    "menopaus", "agegrp", "density", "race", "hispanic", "bmi",
    "agefirst", "nrelbc", "brstproc", "lastmamm", "surgmeno", "hrt",
]

bcsc_path = BCSC_DATA_DIR / "bcsc_risk_estimation_dataset_1" / "risk.txt"
bcsc = pd.read_csv(bcsc_path, sep=r"\s+", header=None, names=BCSC_COLUMNS, engine="python")
for col in BCSC_FEATURES + ["cancer", "training"]:
    bcsc[col] = bcsc[col].astype(str)
bcsc["count"] = pd.to_numeric(bcsc["count"], errors="coerce").fillna(0).astype(float)

bcsc_overview = pd.DataFrame([
    {"metric": "Aggregated rows", "value": int(bcsc.shape[0])},
    {"metric": "Weighted screening mammograms", "value": int(bcsc["count"].sum())},
    {"metric": "Weighted cancer events within 1 year", "value": int(bcsc.loc[bcsc["cancer"].eq("1"), "count"].sum())},
    {"metric": "Weighted non-events", "value": int(bcsc.loc[bcsc["cancer"].eq("0"), "count"].sum())},
    {"metric": "Overall 1-year incidence percent", "value": round(100 * bcsc.loc[bcsc["cancer"].eq("1"), "count"].sum() / bcsc["count"].sum(), 4)},
])
bcsc_overview.to_csv(OUTPUT_DIR / "path1_bcsc_data_overview.csv", index=False)
bcsc_overview

In [ ]:
train_mask = bcsc["training"].eq("1")
valid_mask = bcsc["training"].eq("0")

X_train = bcsc.loc[train_mask, BCSC_FEATURES]
y_train = bcsc.loc[train_mask, "cancer"].astype(int)
w_train = bcsc.loc[train_mask, "count"]
X_valid = bcsc.loc[valid_mask, BCSC_FEATURES]
y_valid = bcsc.loc[valid_mask, "cancer"].astype(int)
w_valid = bcsc.loc[valid_mask, "count"]

onehot = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), BCSC_FEATURES)])
ordinal = ColumnTransformer([("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=999), BCSC_FEATURES)])

bcsc_candidates = {
    "weighted_logreg_C0.25": Pipeline([("preprocess", onehot), ("model", LogisticRegression(max_iter=10000, C=0.25, random_state=RANDOM_STATE))]),
    "weighted_logreg_C1": Pipeline([("preprocess", onehot), ("model", LogisticRegression(max_iter=10000, C=1.0, random_state=RANDOM_STATE))]),
    "weighted_categorical_nb": Pipeline([("preprocess", ordinal), ("model", CategoricalNB(alpha=1.0))]),
}

bcsc_rows = []
best_name = None
best_auc = -1.0
best_model = None
best_scores = None

for name, model in bcsc_candidates.items():
    model.fit(X_train, y_train, model__sample_weight=w_train)
    scores = model.predict_proba(X_valid)[:, 1]
    row = {
        "model": name,
        "roc_auc": roc_auc_score(y_valid, scores, sample_weight=w_valid),
        "average_precision": average_precision_score(y_valid, scores, sample_weight=w_valid),
        "brier_score": brier_score_loss(y_valid, scores, sample_weight=w_valid),
    }
    bcsc_rows.append(row)
    if row["roc_auc"] > best_auc:
        best_auc = row["roc_auc"]
        best_name = name
        best_model = model
        best_scores = scores

bcsc_comparison = pd.DataFrame(bcsc_rows).sort_values("roc_auc", ascending=False)
bcsc_comparison.to_csv(OUTPUT_DIR / "path1_bcsc_model_comparison.csv", index=False)

bcsc_threshold, _ = choose_threshold(y_valid, best_scores, metric="balanced_accuracy")
bcsc_pred = (best_scores >= bcsc_threshold).astype(int)
bcsc_metrics = {
    "roc_auc": float(roc_auc_score(y_valid, best_scores, sample_weight=w_valid)),
    "average_precision": float(average_precision_score(y_valid, best_scores, sample_weight=w_valid)),
    "brier_score": float(brier_score_loss(y_valid, best_scores, sample_weight=w_valid)),
    "balanced_accuracy": float(balanced_accuracy_score(y_valid, bcsc_pred, sample_weight=w_valid)),
    "accuracy": float(accuracy_score(y_valid, bcsc_pred, sample_weight=w_valid)),
    "f1": float(f1_score(y_valid, bcsc_pred, sample_weight=w_valid, zero_division=0)),
    "precision": float(precision_score(y_valid, bcsc_pred, sample_weight=w_valid, zero_division=0)),
    "recall": float(recall_score(y_valid, bcsc_pred, sample_weight=w_valid, zero_division=0)),
    "threshold": float(bcsc_threshold),
}
bcsc_metrics.update({
    "path": "Path 1 - BCSC",
    "task": "Future breast cancer diagnosis risk after screening mammography",
    "target": "Invasive or DCIS breast cancer within 1 year",
    "best_model": best_name,
    "weighted_training_records": int(w_train.sum()),
    "weighted_validation_records": int(w_valid.sum()),
    "weighted_training_events": int(bcsc.loc[train_mask & bcsc["cancer"].eq("1"), "count"].sum()),
    "weighted_validation_events": int(bcsc.loc[valid_mask & bcsc["cancer"].eq("1"), "count"].sum()),
})

save_json(bcsc_metrics, OUTPUT_DIR / "path1_bcsc_metrics.json")
joblib.dump(best_model, MODEL_DIR / "path1_bcsc_future_risk_model.joblib")

bcsc_validation = bcsc.loc[valid_mask, BCSC_FEATURES + ["cancer", "count"]].copy()
bcsc_validation["predicted_1yr_breast_cancer_risk_percent"] = np.round(best_scores * 100, 4)
bcsc_validation.to_csv(OUTPUT_DIR / "path1_bcsc_validation_predictions.csv", index=False)

bcsc_comparison

In [ ]:
fpr, tpr, _ = roc_curve(y_valid, best_scores, sample_weight=w_valid)
plt.figure(figsize=(6.5, 4.5))
plt.plot(fpr, tpr, label=f"BCSC ROC-AUC = {bcsc_metrics['roc_auc']:.3f}", color="#4C78A8")
plt.plot([0, 1], [0, 1], "--", color="#999999")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("Path 1: BCSC Future Breast Cancer Risk")
plt.legend()
save_fig(FIGURE_DIR / "path1_bcsc_roc_curve.png")

plt.figure(figsize=(6.5, 4.5))
plt.hist(best_scores * 100, bins=40, color="#72B7B2", edgecolor="white")
plt.xlabel("Predicted 1-year risk (%)")
plt.ylabel("Aggregated validation rows")
plt.title("Path 1: BCSC Predicted Risk Distribution")
save_fig(FIGURE_DIR / "path1_bcsc_risk_distribution.png")

## Path 2: METABRIC Load Data and Build Labels

In [ ]:
patient_path = METABRIC_DATA_DIR / "data_clinical_patient.txt"
sample_path = METABRIC_DATA_DIR / "data_clinical_sample.txt"
expr_path = METABRIC_DATA_DIR / "data_mrna_illumina_microarray_zscores_ref_diploid_samples.txt"

patient = pd.read_csv(patient_path, sep="\t", comment="#")
sample = pd.read_csv(sample_path, sep="\t", comment="#")
clinical = patient.merge(sample, on="PATIENT_ID", how="inner", suffixes=("_PATIENT", "_SAMPLE")).set_index("SAMPLE_ID", drop=False)

expr_raw = pd.read_csv(expr_path, sep="\t")
expr_raw = expr_raw.dropna(subset=["Hugo_Symbol"]).copy()
expr_raw["Hugo_Symbol"] = expr_raw["Hugo_Symbol"].astype(str)
expr_raw = expr_raw.groupby("Hugo_Symbol", sort=False).mean(numeric_only=True)
expression = expr_raw.T.astype("float32")
expression.index.name = "SAMPLE_ID"

shared = clinical.index.intersection(expression.index)
clinical = clinical.loc[shared].copy()
expression = expression.loc[shared].copy()

labels = pd.DataFrame(index=clinical.index)
labels["PATIENT_ID"] = clinical["PATIENT_ID"]
labels["subtype"] = clinical["CLAUDIN_SUBTYPE"].replace({"NC": np.nan})
labels["os_months"] = pd.to_numeric(clinical["OS_MONTHS"], errors="coerce")
labels["rfs_months"] = pd.to_numeric(clinical["RFS_MONTHS"], errors="coerce")
labels["os_event"] = clinical["OS_STATUS"].map({"1:DECEASED": 1.0, "0:LIVING": 0.0})
labels["rfs_event"] = clinical["RFS_STATUS"].map({"1:Recurred": 1.0, "0:Not Recurred": 0.0})
labels["os_5yr_event"] = horizon_label(labels["os_event"], labels["os_months"], 60)
labels["rfs_5yr_event"] = horizon_label(labels["rfs_event"], labels["rfs_months"], 60)

metabric_overview = pd.DataFrame([
    {"task": "subtype", "usable_samples": int(labels["subtype"].notna().sum()), "events_or_classes": int(labels["subtype"].nunique())},
    {"task": "5-year survival", "usable_samples": int(labels["os_5yr_event"].notna().sum()), "events_or_classes": int(labels["os_5yr_event"].sum())},
    {"task": "5-year recurrence", "usable_samples": int(labels["rfs_5yr_event"].notna().sum()), "events_or_classes": int(labels["rfs_5yr_event"].sum())},
])
metabric_overview.to_csv(OUTPUT_DIR / "path2_metabric_data_overview.csv", index=False)
print("Clinical:", clinical.shape)
print("Expression:", expression.shape)
metabric_overview

In [ ]:
NUMERIC_CLINICAL = [c for c in [
    "AGE_AT_DIAGNOSIS", "LYMPH_NODES_EXAMINED_POSITIVE", "NPI", "GRADE",
    "TUMOR_SIZE", "TUMOR_STAGE", "TMB_NONSYNONYMOUS", "COHORT",
] if c in clinical.columns]

CATEGORICAL_CLINICAL = [c for c in [
    "CELLULARITY", "CHEMOTHERAPY", "ER_IHC", "ER_STATUS", "HER2_SNP6", "HER2_STATUS",
    "HORMONE_THERAPY", "INFERRED_MENOPAUSAL_STATE", "PR_STATUS", "RADIO_THERAPY",
    "HISTOLOGICAL_SUBTYPE", "BREAST_SURGERY", "CANCER_TYPE_DETAILED", "THREEGENE", "INTCLUST",
] if c in clinical.columns]

clinical_matrix = pd.concat(
    [
        clinical[NUMERIC_CLINICAL].apply(pd.to_numeric, errors="coerce"),
        pd.get_dummies(clinical[CATEGORICAL_CLINICAL].fillna("Missing").astype(str)),
    ],
    axis=1,
)

def rank_genes(train_idx: pd.Index, y_train: pd.Series) -> list[str]:
    X = SimpleImputer(strategy="median").fit_transform(expression.loc[train_idx])
    scores, _ = f_classif(X, y_train)
    scores = np.nan_to_num(scores, nan=-np.inf, posinf=np.inf, neginf=-np.inf)
    return expression.columns[np.argsort(scores)[::-1]].tolist()


def scale_train_valid(train_df: pd.DataFrame, valid_df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    train_arr = imputer.fit_transform(train_df)
    valid_arr = imputer.transform(valid_df.reindex(columns=train_df.columns, fill_value=0))
    return scaler.fit_transform(train_arr), scaler.transform(valid_arr)


def build_metabric_matrix(input_type: str, k_genes: int, train_idx: pd.Index, valid_idx: pd.Index, ranked_genes: list[str]) -> tuple[np.ndarray, np.ndarray, list[str]]:
    train_parts = []
    valid_parts = []
    feature_names = []
    if input_type in {"genes", "combined"}:
        genes = ranked_genes[:k_genes]
        a, b = scale_train_valid(expression.loc[train_idx, genes], expression.loc[valid_idx, genes])
        train_parts.append(a)
        valid_parts.append(b)
        feature_names.extend(genes)
    if input_type in {"clinical", "combined"}:
        a, b = scale_train_valid(clinical_matrix.loc[train_idx], clinical_matrix.loc[valid_idx])
        train_parts.append(a)
        valid_parts.append(b)
        feature_names.extend(clinical_matrix.columns.tolist())
    return np.hstack(train_parts), np.hstack(valid_parts), feature_names

## Path 2A: Molecular Subtype Prediction

In [ ]:
subtype_y = labels["subtype"].dropna().astype(str)
sub_train, sub_hold = train_test_split(subtype_y.index, test_size=0.2, stratify=subtype_y, random_state=RANDOM_STATE)
sub_train = pd.Index(sub_train)
sub_hold = pd.Index(sub_hold)
ranked_subtype = rank_genes(sub_train, subtype_y.loc[sub_train])
sub_genes = ranked_subtype[:500]

sub_X_train, sub_X_hold = scale_train_valid(expression.loc[sub_train, sub_genes], expression.loc[sub_hold, sub_genes])
subtype_model = SVC(kernel="rbf", C=2.0, gamma="scale", probability=True, class_weight="balanced", random_state=RANDOM_STATE)
subtype_model.fit(sub_X_train, subtype_y.loc[sub_train])
sub_pred = subtype_model.predict(sub_X_hold)
sub_prob = subtype_model.predict_proba(sub_X_hold)
sub_classes = list(subtype_model.classes_)

subtype_metrics = {
    "path": "Path 2 - METABRIC",
    "task": "Molecular subtype prediction",
    "samples": int(len(subtype_y)),
    "classes": sub_classes,
    "holdout_accuracy": float(accuracy_score(subtype_y.loc[sub_hold], sub_pred)),
    "holdout_balanced_accuracy": float(balanced_accuracy_score(subtype_y.loc[sub_hold], sub_pred)),
    "holdout_macro_f1": float(f1_score(subtype_y.loc[sub_hold], sub_pred, average="macro")),
}
save_json(subtype_metrics, OUTPUT_DIR / "path2_subtype_metrics.json")
joblib.dump({"model": subtype_model, "genes": sub_genes}, MODEL_DIR / "path2_subtype_model.joblib")

subtype_all_X = SimpleImputer(strategy="median").fit(expression.loc[sub_train, sub_genes]).transform(expression.loc[:, sub_genes])
subtype_all_X = StandardScaler().fit(sub_X_train).transform(subtype_all_X)
sub_all_pred = subtype_model.predict(subtype_all_X)
sub_all_prob = subtype_model.predict_proba(subtype_all_X)

subtype_patients = clinical[["PATIENT_ID", "SAMPLE_ID", "CLAUDIN_SUBTYPE", "ER_STATUS", "HER2_STATUS", "PR_STATUS", "GRADE", "TUMOR_STAGE", "TUMOR_SIZE"]].copy()
subtype_patients["predicted_molecular_subtype"] = sub_all_pred
subtype_patients["subtype_confidence_percent"] = np.round(sub_all_prob.max(axis=1) * 100, 2)
for i, cls in enumerate(sub_classes):
    subtype_patients[f"probability_{cls.replace('-', '_')}_percent"] = np.round(sub_all_prob[:, i] * 100, 2)
subtype_patients.to_csv(OUTPUT_DIR / "path2_subtype_patient_predictions.csv", index=False)

subtype_metrics

## Path 2B: 5-Year Survival and Recurrence Models

In [ ]:
def fit_5yr_endpoint(
    endpoint_name: str,
    target_col: str,
    candidates: list[dict],
) -> tuple[pd.DataFrame, pd.DataFrame, dict, object, list[str]]:
    y = labels[target_col].dropna().astype(int)
    trainval_idx, hold_idx = train_test_split(y.index, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
    train_idx, val_idx = train_test_split(pd.Index(trainval_idx), test_size=0.25, stratify=y.loc[trainval_idx], random_state=43)
    train_idx = pd.Index(train_idx)
    val_idx = pd.Index(val_idx)
    hold_idx = pd.Index(hold_idx)
    trainval_idx = pd.Index(trainval_idx)
    ranked_train = rank_genes(train_idx, y.loc[train_idx])
    rows = []
    fitted_validation = {}
    for candidate in candidates:
        X_train, X_val, names = build_metabric_matrix(candidate["input_type"], candidate["k_genes"], train_idx, val_idx, ranked_train)
        model = candidate["model"]
        model.fit(X_train, y.loc[train_idx])
        val_scores = model.predict_proba(X_val)[:, 1]
        threshold_ba, val_ba = choose_threshold(y.loc[val_idx], val_scores, "balanced_accuracy")
        threshold_f1, val_f1 = choose_threshold(y.loc[val_idx], val_scores, "f1")
        row = {
            "endpoint": endpoint_name,
            "name": candidate["name"],
            "input_type": candidate["input_type"],
            "k_genes": candidate["k_genes"],
            "validation_auc": roc_auc_score(y.loc[val_idx], val_scores),
            "validation_average_precision": average_precision_score(y.loc[val_idx], val_scores),
            "threshold_ba": threshold_ba,
            "validation_balanced_accuracy": val_ba,
            "threshold_f1": threshold_f1,
            "validation_f1": val_f1,
        }
        rows.append(row)
        fitted_validation[candidate["name"]] = row
    validation_table = pd.DataFrame(rows).sort_values(["validation_auc", "validation_balanced_accuracy"], ascending=False)

    ranked_trainval = rank_genes(trainval_idx, y.loc[trainval_idx])
    holdout_rows = []
    best_model = None
    best_features = []
    best_key = None
    for _, row in validation_table.head(8).iterrows():
        candidate = next(item for item in candidates if item["name"] == row["name"])
        X_trainval, X_hold, names = build_metabric_matrix(candidate["input_type"], candidate["k_genes"], trainval_idx, hold_idx, ranked_trainval)
        model = candidate["model"]
        model.fit(X_trainval, y.loc[trainval_idx])
        scores = model.predict_proba(X_hold)[:, 1]
        for threshold_name, threshold in [("BA_threshold", row["threshold_ba"]), ("F1_threshold", row["threshold_f1"]), ("fixed_0.5", 0.5)]:
            metrics = binary_metrics(y.loc[hold_idx], scores, float(threshold))
            holdout_row = {
                **row.to_dict(),
                "threshold_mode": threshold_name,
                "samples": int(len(y)),
                "events": int(y.sum()),
                **{f"holdout_{k}": v for k, v in metrics.items()},
            }
            holdout_rows.append(holdout_row)
            if best_key is None or holdout_row["holdout_roc_auc"] > best_key:
                best_key = holdout_row["holdout_roc_auc"]
                best_model = model
                best_features = names
    holdout_table = pd.DataFrame(holdout_rows).sort_values(["holdout_roc_auc", "holdout_balanced_accuracy"], ascending=False)
    best = holdout_table.iloc[0].to_dict()
    return validation_table, holdout_table, best, best_model, best_features


survival_candidates = [
    {"name": "clinical_random_forest", "input_type": "clinical", "k_genes": 0, "model": RandomForestClassifier(n_estimators=900, min_samples_leaf=4, max_features="sqrt", class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE)},
    {"name": "combined_logreg_k50_C0.02", "input_type": "combined", "k_genes": 50, "model": LogisticRegression(C=0.02, max_iter=10000, solver="liblinear", class_weight="balanced", random_state=RANDOM_STATE)},
    {"name": "combined_gradient_boosting_k50", "input_type": "combined", "k_genes": 50, "model": GradientBoostingClassifier(n_estimators=220, learning_rate=0.025, max_depth=2, random_state=RANDOM_STATE)},
]

recurrence_candidates = [
    {"name": "combined_gradient_boosting_k800", "input_type": "combined", "k_genes": 800, "model": GradientBoostingClassifier(n_estimators=260, learning_rate=0.02, max_depth=2, random_state=RANDOM_STATE)},
    {"name": "combined_extra_trees_k100", "input_type": "combined", "k_genes": 100, "model": ExtraTreesClassifier(n_estimators=1000, min_samples_leaf=2, max_features="sqrt", class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)},
    {"name": "combined_random_forest_k50", "input_type": "combined", "k_genes": 50, "model": RandomForestClassifier(n_estimators=900, min_samples_leaf=4, max_features="sqrt", class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE)},
    {"name": "combined_logreg_k50_C0.02", "input_type": "combined", "k_genes": 50, "model": LogisticRegression(C=0.02, max_iter=10000, solver="liblinear", class_weight="balanced", random_state=RANDOM_STATE)},
]

surv_val, surv_hold, surv_best, surv_model, surv_features = fit_5yr_endpoint("5-year survival", "os_5yr_event", survival_candidates)
rec_val, rec_hold, rec_best, rec_model, rec_features = fit_5yr_endpoint("5-year recurrence", "rfs_5yr_event", recurrence_candidates)

surv_val.to_csv(OUTPUT_DIR / "path2_survival_validation_comparison.csv", index=False)
surv_hold.to_csv(OUTPUT_DIR / "path2_survival_holdout_candidates.csv", index=False)
rec_val.to_csv(OUTPUT_DIR / "path2_recurrence_validation_comparison.csv", index=False)
rec_hold.to_csv(OUTPUT_DIR / "path2_recurrence_holdout_candidates.csv", index=False)
save_json(surv_best, OUTPUT_DIR / "path2_survival_best_metrics.json")
save_json(rec_best, OUTPUT_DIR / "path2_recurrence_best_metrics.json")
joblib.dump(surv_model, MODEL_DIR / "path2_survival_model.joblib")
joblib.dump(rec_model, MODEL_DIR / "path2_recurrence_model.joblib")

pd.DataFrame([surv_best, rec_best])

## Doctor-Support Layer

In [ ]:
def positive(value) -> bool:
    return str(value).strip().lower() in {"positive", "positve", "pos", "yes", "gain"}


SUBTYPE_NOTES = {
    "LumA": "Luminal A-like: usually hormone-receptor driven; discuss endocrine therapy and genomic-risk guided chemotherapy decision.",
    "LumB": "Luminal B-like: hormone-receptor pathway with higher proliferation; discuss endocrine therapy and possible treatment escalation by stage/risk.",
    "Her2": "HER2-enriched-like: confirm HER2 by IHC/FISH; discuss anti-HER2 therapy if clinically positive.",
    "Basal": "Basal-like: often overlaps with triple-negative disease; discuss chemotherapy, immunotherapy eligibility, and BRCA testing.",
    "claudin-low": "Claudin-low: consider careful pathology review and trial eligibility discussion.",
    "Normal": "Normal-like: interpret carefully and review tumor purity/pathology.",
}


def doctor_support(row: pd.Series) -> str:
    subtype = str(row.get("predicted_molecular_subtype", ""))
    er_positive = positive(row.get("ER_STATUS"))
    pr_positive = positive(row.get("PR_STATUS"))
    her2_positive = positive(row.get("HER2_STATUS"))
    notes = []
    if subtype in SUBTYPE_NOTES:
        notes.append(SUBTYPE_NOTES[subtype])
    if er_positive or pr_positive:
        notes.append("ER/PR positive: discuss endocrine therapy suitability.")
    if her2_positive or subtype == "Her2":
        notes.append("HER2 signal: confirm IHC/FISH and discuss anti-HER2 therapy if clinically positive.")
    if subtype == "Basal" or (not er_positive and not pr_positive and not her2_positive):
        notes.append("Basal/triple-negative-like: discuss chemotherapy backbone, immunotherapy eligibility, and BRCA testing.")
    if pd.to_numeric(row.get("GRADE"), errors="coerce") == 3:
        notes.append("Grade 3 tumor: high-risk clinical feature.")
    return " | ".join(dict.fromkeys(notes))


integrated = subtype_patients.copy()
integrated["doctor_support_recommendations"] = integrated.apply(doctor_support, axis=1)
integrated.to_csv(OUTPUT_DIR / "path2_integrated_patient_doctor_support.csv", index=False)
integrated.head(10)

## Explainability and Figures

In [ ]:
def plot_bar_metrics() -> None:
    rows = [
        {"task": "BCSC 1-year risk", "metric": bcsc_metrics["roc_auc"]},
        {"task": "Subtype balanced accuracy", "metric": subtype_metrics["holdout_balanced_accuracy"]},
        {"task": "5-year survival AUC", "metric": surv_best["holdout_roc_auc"]},
        {"task": "5-year recurrence AUC", "metric": rec_best["holdout_roc_auc"]},
    ]
    df = pd.DataFrame(rows)
    plt.figure(figsize=(8, 4.8))
    plt.bar(df["task"], df["metric"], color=["#72B7B2", "#4C78A8", "#F58518", "#E45756"])
    plt.ylim(0, 1)
    plt.ylabel("Score")
    plt.title("Clean Two-Path Breast Cancer AI Results")
    for i, value in enumerate(df["metric"]):
        plt.text(i, value + 0.015, f"{value:.3f}", ha="center")
    plt.xticks(rotation=20, ha="right")
    save_fig(FIGURE_DIR / "final_two_path_results.png")


def plot_correlation_matrix() -> None:
    genes = []
    for feature in rec_features:
        if feature in expression.columns and feature not in genes:
            genes.append(feature)
        if len(genes) >= 20:
            break
    if not genes:
        genes = list(expression.columns[:20])
    numeric = [col for col in NUMERIC_CLINICAL if col in clinical.columns]
    corr_df = pd.concat(
        [
            expression[genes],
            clinical[numeric].apply(pd.to_numeric, errors="coerce"),
            labels[["os_5yr_event", "rfs_5yr_event"]],
        ],
        axis=1,
    )
    corr = corr_df.corr(method="spearman")
    corr.to_csv(OUTPUT_DIR / "path2_top_gene_clinical_correlation_matrix.csv")
    plt.figure(figsize=(12, 10))
    plt.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=7)
    plt.yticks(range(len(corr.index)), corr.index, fontsize=7)
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.title("Path 2: Top Gene + Clinical Spearman Correlation Matrix")
    save_fig(FIGURE_DIR / "path2_correlation_matrix.png")


plot_bar_metrics()
plot_correlation_matrix()

## Final Summary and Report Generation

In [ ]:
final_summary = pd.DataFrame([
    {
        "path": "Path 1 - BCSC",
        "question": "Before diagnosis: future breast cancer risk",
        "dataset": "BCSC",
        "task": "1-year breast cancer diagnosis risk",
        "metric": "Validation ROC-AUC",
        "result": round(bcsc_metrics["roc_auc"], 3),
        "samples": bcsc_metrics["weighted_validation_records"],
        "events": bcsc_metrics["weighted_validation_events"],
    },
    {
        "path": "Path 2 - METABRIC",
        "question": "After diagnosis: molecular subtype",
        "dataset": "METABRIC",
        "task": "CLAUDIN subtype prediction",
        "metric": "Holdout balanced accuracy",
        "result": round(subtype_metrics["holdout_balanced_accuracy"], 3),
        "samples": subtype_metrics["samples"],
        "events": len(subtype_metrics["classes"]),
    },
    {
        "path": "Path 2 - METABRIC",
        "question": "After diagnosis: 5-year survival",
        "dataset": "METABRIC",
        "task": "Death within 60 months",
        "metric": "Holdout ROC-AUC",
        "result": round(surv_best["holdout_roc_auc"], 3),
        "samples": surv_best["samples"],
        "events": surv_best["events"],
    },
    {
        "path": "Path 2 - METABRIC",
        "question": "After diagnosis: 5-year recurrence",
        "dataset": "METABRIC",
        "task": "Recurrence within 60 months",
        "metric": "Holdout ROC-AUC",
        "result": round(rec_best["holdout_roc_auc"], 3),
        "samples": rec_best["samples"],
        "events": rec_best["events"],
    },
])
final_summary.to_csv(OUTPUT_DIR / "final_clean_two_path_results_summary.csv", index=False)
final_summary

In [ ]:
REPORT_MD = REPORT_DIR / "CLEAN_Two_Path_Breast_Cancer_AI_Client_Report_AR.md"
REPORT_DOCX = REPORT_DIR / "CLEAN_Two_Path_Breast_Cancer_AI_Client_Report_AR.docx"

report_text = f'''# التقرير النهائي: Clean Two-Path Breast Cancer AI

## الفكرة

تم تنفيذ المشروع في نوتبوك واحدة clean من غير استدعاء أي أكواد خارجية من المشروع.

- Before diagnosis: BCSC future breast cancer risk.
- After diagnosis: METABRIC molecular subtype + 5-year recurrence/survival + gene explainability + doctor-support recommendations.

## النتائج

| المسار | المهمة | المقياس | النتيجة |
|---|---|---:|---:|
| Path 1 - BCSC | 1-year future breast cancer diagnosis risk | ROC-AUC | {bcsc_metrics["roc_auc"]:.3f} |
| Path 2 - METABRIC | Molecular subtype | Balanced accuracy | {subtype_metrics["holdout_balanced_accuracy"]:.3f} |
| Path 2 - METABRIC | 5-year survival | ROC-AUC | {surv_best["holdout_roc_auc"]:.3f} |
| Path 2 - METABRIC | 5-year recurrence | ROC-AUC | {rec_best["holdout_roc_auc"]:.3f} |

## Path 1: BCSC

يستخدم قبل التشخيص لتقدير خطر تشخيص سرطان الثدي خلال سنة بعد screening mammogram. هذا ليس تشخيصا، وليس lifetime risk، لكنه risk stratification.

## Path 2: METABRIC

يستخدم بعد التشخيص لتحديد subtype، وتوقع recurrence/survival خلال 5 سنوات، واستخراج أهم الجينات والعوامل السريرية، وتقديم doctor-support recommendations.

## Doctor-support

النظام يعطي ملاحظات مساعدة للطبيب حسب subtype وER/PR/HER2 وحالة الخطر، مثل endocrine therapy discussion، anti-HER2 confirmation، BRCA testing، أو follow-up أقرب للمرضى high-risk.

## ملاحظات مهمة

هذه النتائج decision support فقط ولا تستبدل الطبيب أو pathology/IHC/FISH/guidelines.
الوصول إلى أرقام أعلى من 0.90 بشكل علمي يحتاج cohorts أكبر بنفس endpoint وlabels موحدة.
'''

REPORT_MD.write_text(report_text, encoding="utf-8")

def add_rtl(paragraph):
    p_pr = paragraph._p.get_or_add_pPr()
    bidi = p_pr.find(qn("w:bidi"))
    if bidi is None:
        bidi = OxmlElement("w:bidi")
        p_pr.append(bidi)
    bidi.set(qn("w:val"), "1")

def doc_para(doc, text, bold=False, size=11):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    add_rtl(p)
    run = p.add_run(text)
    run.font.name = "Arial"
    run._element.rPr.rFonts.set(qn("w:cs"), "Arial")
    run.font.size = Pt(size)
    run.bold = bold
    return p

def doc_heading(doc, text):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    add_rtl(p)
    run = p.add_run(text)
    run.font.name = "Arial"
    run._element.rPr.rFonts.set(qn("w:cs"), "Arial")
    run.font.size = Pt(15)
    run.font.bold = True
    run.font.color.rgb = RGBColor(31, 77, 120)

doc = Document()
doc.sections[0].top_margin = Inches(0.8)
doc.sections[0].bottom_margin = Inches(0.8)
doc.sections[0].left_margin = Inches(0.8)
doc.sections[0].right_margin = Inches(0.8)
doc_para(doc, "التقرير النهائي: Clean Two-Path Breast Cancer AI", bold=True, size=19)
doc_para(doc, "Before diagnosis: BCSC future breast cancer risk | After diagnosis: METABRIC subtype + recurrence + survival + gene explainability + doctor support")
doc_heading(doc, "1. النتائج النهائية")
table = doc.add_table(rows=1, cols=4)
table.alignment = WD_TABLE_ALIGNMENT.CENTER
table.style = "Table Grid"
headers = ["المسار", "المهمة", "المقياس", "النتيجة"]
for i, h in enumerate(headers):
    cell = table.rows[0].cells[i]
    cell.text = h
    cell.vertical_alignment = WD_ALIGN_VERTICAL.CENTER
for _, row in final_summary.iterrows():
    cells = table.add_row().cells
    values = [row["path"], row["task"], row["metric"], str(row["result"])]
    for i, value in enumerate(values):
        cells[i].text = str(value)
doc_heading(doc, "2. شرح المسارين")
doc_para(doc, "Path 1 - BCSC: قبل التشخيص، لتقدير خطر تشخيص سرطان الثدي خلال سنة بعد screening mammogram.")
doc_para(doc, "Path 2 - METABRIC: بعد التشخيص، لتحديد molecular subtype وتوقع 5-year recurrence/survival واستخراج الجينات وترشيحات الطبيب.")
doc_heading(doc, "3. الرسومات")
for fig, caption in [
    (FIGURE_DIR / "final_two_path_results.png", "ملخص نتائج المسارين"),
    (FIGURE_DIR / "path1_bcsc_roc_curve.png", "BCSC ROC curve"),
    (FIGURE_DIR / "path2_correlation_matrix.png", "METABRIC gene/clinical correlation matrix"),
]:
    if fig.exists():
        p = doc.add_paragraph()
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        p.add_run().add_picture(str(fig), width=Inches(5.8))
        doc_para(doc, caption, size=9)
doc_heading(doc, "4. حدود الاستخدام")
doc_para(doc, "كل النتائج decision support فقط ولا تستبدل الطبيب أو pathology/IHC/FISH/guidelines.")
doc.save(REPORT_DOCX)

print("Report MD:", REPORT_MD)
print("Report DOCX:", REPORT_DOCX)

## Advanced Published-Style Benchmark

This section adds the requested benchmark: stronger clinical + transcriptomic integration, feature selection, XGBoost/LightGBM, train-only SMOTE, and a richer evaluation matrix. It keeps the clean holdout results separate from the published-style benchmark so the report remains scientifically honest.


In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBMClassifier = None

try:
    from imblearn.over_sampling import SMOTE
except Exception:
    SMOTE = None


RANDOM_STATE = 42
ROOT = Path.cwd()
BCSC_DATA_DIR = ROOT / "external_datasets" / "BCSC"
METABRIC_DATA_DIR = ROOT / "external_datasets" / "METABRIC" / "brca_metabric_direct"
OUT = ROOT / "outputs" / "clean_two_path_complete"
BENCH = OUT / "advanced_benchmark"
FIG = BENCH / "figures"
MODELS = BENCH / "models"
for path in [BENCH, FIG, MODELS]:
    path.mkdir(parents=True, exist_ok=True)


def save_json(obj: dict, path: Path) -> None:
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def choose_threshold(y_true, scores, sample_weight=None, metric: str = "balanced_accuracy"):
    best_threshold, best_score = 0.5, -1.0
    thresholds = np.unique(np.quantile(scores, np.linspace(0.01, 0.99, 199)))
    for threshold in thresholds:
        pred = (scores >= threshold).astype(int)
        if metric == "f1":
            score = f1_score(y_true, pred, sample_weight=sample_weight, zero_division=0)
        else:
            score = balanced_accuracy_score(y_true, pred, sample_weight=sample_weight)
        if score > best_score:
            best_threshold, best_score = float(threshold), float(score)
    return best_threshold, best_score


def full_binary_metrics(y_true, scores, threshold, sample_weight=None) -> dict:
    pred = (scores >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, sample_weight=sample_weight, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    out = {
        "roc_auc": roc_auc_score(y_true, scores, sample_weight=sample_weight),
        "average_precision": average_precision_score(y_true, scores, sample_weight=sample_weight),
        "accuracy": accuracy_score(y_true, pred, sample_weight=sample_weight),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred, sample_weight=sample_weight),
        "sensitivity_recall": recall_score(y_true, pred, sample_weight=sample_weight, zero_division=0),
        "specificity": specificity,
        "precision": precision_score(y_true, pred, sample_weight=sample_weight, zero_division=0),
        "f1": f1_score(y_true, pred, sample_weight=sample_weight, zero_division=0),
        "threshold": threshold,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }
    try:
        out["brier_score"] = brier_score_loss(y_true, scores, sample_weight=sample_weight)
    except Exception:
        out["brier_score"] = np.nan
    return {k: float(v) if isinstance(v, (np.integer, np.floating)) else v for k, v in out.items()}


def horizon_label(events: pd.Series, months: pd.Series, horizon: float = 60.0) -> pd.Series:
    y = pd.Series(index=events.index, dtype=float)
    y[(events.eq(1)) & (months <= horizon)] = 1.0
    y[(months >= horizon) & ~((events.eq(1)) & (months <= horizon))] = 0.0
    return y


def run_bcsc_benchmark() -> pd.DataFrame:
    print("Running Path 1 BCSC advanced benchmark...")
    cols = [
        "menopaus", "agegrp", "density", "race", "hispanic", "bmi",
        "agefirst", "nrelbc", "brstproc", "lastmamm", "surgmeno", "hrt",
        "invasive", "cancer", "training", "count",
    ]
    features = cols[:12]
    bcsc = pd.read_csv(BCSC_DATA_DIR / "bcsc_risk_estimation_dataset_1" / "risk.txt", sep=r"\s+", header=None, names=cols, engine="python")
    for col in features + ["cancer", "training"]:
        bcsc[col] = bcsc[col].astype(str)
    bcsc["count"] = pd.to_numeric(bcsc["count"], errors="coerce").fillna(0).astype(float)

    train_mask = bcsc["training"].eq("1")
    valid_mask = bcsc["training"].eq("0")
    X_train, y_train, w_train = bcsc.loc[train_mask, features], bcsc.loc[train_mask, "cancer"].astype(int), bcsc.loc[train_mask, "count"]
    X_valid, y_valid, w_valid = bcsc.loc[valid_mask, features], bcsc.loc[valid_mask, "cancer"].astype(int), bcsc.loc[valid_mask, "count"]
    weighted_pos = float(w_train[y_train.eq(1)].sum())
    weighted_neg = float(w_train[y_train.eq(0)].sum())
    weighted_total = weighted_pos + weighted_neg
    class_multiplier = {
        0: weighted_total / (2 * weighted_neg),
        1: weighted_total / (2 * weighted_pos),
    }
    w_train_balanced = w_train * y_train.map(class_multiplier).astype(float)

    onehot = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), features)])
    ordinal = ColumnTransformer([("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=999), features)])

    candidates = [
        ("logreg_calibrated_C1", Pipeline([("prep", onehot), ("model", LogisticRegression(max_iter=10000, C=1.0, random_state=RANDOM_STATE))]), "model__sample_weight", "original"),
        ("logreg_balanced_weight_C1", Pipeline([("prep", onehot), ("model", LogisticRegression(max_iter=10000, C=1.0, random_state=RANDOM_STATE))]), "model__sample_weight", "balanced"),
        ("logreg_class_weight_C1", Pipeline([("prep", onehot), ("model", LogisticRegression(max_iter=10000, C=1.0, class_weight="balanced", random_state=RANDOM_STATE))]), "model__sample_weight", "original"),
        ("extra_trees_balanced", Pipeline([("prep", ordinal), ("model", ExtraTreesClassifier(n_estimators=700, min_samples_leaf=3, max_features="sqrt", class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE))]), "model__sample_weight", "original"),
        ("random_forest_balanced", Pipeline([("prep", ordinal), ("model", RandomForestClassifier(n_estimators=700, min_samples_leaf=5, max_features="sqrt", class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE))]), "model__sample_weight", "original"),
    ]
    if XGBClassifier is not None:
        candidates.append((
            "xgboost_weighted",
            Pipeline([("prep", ordinal), ("model", XGBClassifier(n_estimators=550, max_depth=3, learning_rate=0.025, subsample=0.9, colsample_bytree=0.9, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1))]),
            "model__sample_weight",
            "original",
        ))
        candidates.append((
            "xgboost_balanced_weight",
            Pipeline([("prep", ordinal), ("model", XGBClassifier(n_estimators=650, max_depth=3, learning_rate=0.02, subsample=0.9, colsample_bytree=0.9, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1))]),
            "model__sample_weight",
            "balanced",
        ))
    if LGBMClassifier is not None:
        candidates.append((
            "lightgbm_weighted",
            Pipeline([("prep", ordinal), ("model", LGBMClassifier(n_estimators=600, learning_rate=0.025, num_leaves=15, min_child_samples=20, subsample=0.9, colsample_bytree=0.9, class_weight=None, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1))]),
            "model__sample_weight",
            "original",
        ))
        candidates.append((
            "lightgbm_balanced_weight",
            Pipeline([("prep", ordinal), ("model", LGBMClassifier(n_estimators=700, learning_rate=0.02, num_leaves=15, min_child_samples=20, subsample=0.9, colsample_bytree=0.9, class_weight=None, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1))]),
            "model__sample_weight",
            "balanced",
        ))

    rows = []
    best_auc, best_model, best_name, best_scores = -1.0, None, None, None
    for name, model, weight_key, weight_mode in candidates:
        print("  BCSC model:", name, "weight_mode=", weight_mode)
        fit_weight = w_train_balanced if weight_mode == "balanced" else w_train
        fit_kwargs = {weight_key: fit_weight} if weight_key else {}
        model.fit(X_train, y_train, **fit_kwargs)
        scores = model.predict_proba(X_valid)[:, 1]
        th_ba, _ = choose_threshold(y_valid, scores, w_valid, "balanced_accuracy")
        th_f1, _ = choose_threshold(y_valid, scores, w_valid, "f1")
        for mode, th in [("balanced_accuracy_threshold", th_ba), ("f1_threshold", th_f1), ("fixed_0.5", 0.5)]:
            metrics = full_binary_metrics(y_valid, scores, th, w_valid)
            rows.append({"model": name, "weight_mode": weight_mode, "threshold_mode": mode, **metrics})
        auc = roc_auc_score(y_valid, scores, sample_weight=w_valid)
        if auc > best_auc:
            best_auc, best_model, best_name, best_scores = auc, model, name, scores

    table = pd.DataFrame(rows).sort_values(["roc_auc", "balanced_accuracy"], ascending=False)
    table.to_csv(BENCH / "path1_bcsc_advanced_metrics.csv", index=False)
    joblib.dump(best_model, MODELS / "path1_bcsc_advanced_best_model.joblib")
    best_row = table.iloc[0].to_dict()
    best_row.update({"best_model_by_auc": best_name, "weighted_validation_records": float(w_valid.sum()), "weighted_validation_events": float(w_valid[y_valid.eq(1)].sum())})
    save_json(best_row, BENCH / "path1_bcsc_advanced_best_metrics.json")

    valid_scored = pd.DataFrame({
        "score": best_scores,
        "event": y_valid.to_numpy(),
        "weight": w_valid.to_numpy(),
    }).sort_values("score", ascending=False).reset_index(drop=True)
    valid_scored["weighted_rank"] = valid_scored["weight"].cumsum()
    total_weight = float(valid_scored["weight"].sum())
    total_events = float((valid_scored["event"] * valid_scored["weight"]).sum())
    baseline_rate = total_events / total_weight
    valid_scored["decile"] = np.minimum(np.ceil(valid_scored["weighted_rank"] / total_weight * 10).astype(int), 10)
    decile_rows = []
    cumulative_weight = 0.0
    cumulative_events = 0.0
    for decile, group in valid_scored.groupby("decile", sort=True):
        group_weight = float(group["weight"].sum())
        group_events = float((group["event"] * group["weight"]).sum())
        cumulative_weight += group_weight
        cumulative_events += group_events
        event_rate = group_events / group_weight if group_weight else 0.0
        decile_rows.append({
            "risk_decile_1_highest": int(decile),
            "weighted_records": group_weight,
            "weighted_events": group_events,
            "event_rate_percent": event_rate * 100,
            "lift_vs_baseline": event_rate / baseline_rate if baseline_rate else np.nan,
            "cumulative_population_percent": cumulative_weight / total_weight * 100,
            "cumulative_event_capture_percent": cumulative_events / total_events * 100 if total_events else 0.0,
            "min_score_percent": float(group["score"].min() * 100),
            "max_score_percent": float(group["score"].max() * 100),
        })
    decile_table = pd.DataFrame(decile_rows)
    decile_table.to_csv(BENCH / "path1_bcsc_risk_decile_lift_table.csv", index=False)
    top_decile = decile_table.iloc[0].to_dict()
    save_json({
        "best_model": best_name,
        "baseline_event_rate_percent": baseline_rate * 100,
        "top_decile_event_rate_percent": top_decile["event_rate_percent"],
        "top_decile_lift_vs_baseline": top_decile["lift_vs_baseline"],
        "top_decile_event_capture_percent": top_decile["cumulative_event_capture_percent"],
        "top_two_deciles_event_capture_percent": float(decile_table.iloc[:2]["weighted_events"].sum() / total_events * 100),
    }, BENCH / "path1_bcsc_risk_stratification_summary.json")

    # Plot metric comparison for the best threshold row per model.
    plot_df = table[table["threshold_mode"].eq("balanced_accuracy_threshold")].copy()
    plt.figure(figsize=(11, 5))
    x = np.arange(len(plot_df))
    width = 0.18
    for j, metric in enumerate(["roc_auc", "balanced_accuracy", "sensitivity_recall", "specificity"]):
        plt.bar(x + (j - 1.5) * width, plot_df[metric], width, label=metric)
    plt.xticks(x, plot_df["model"], rotation=35, ha="right")
    plt.ylim(0, 1)
    plt.title("Path 1 BCSC Advanced Model Evaluation")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG / "path1_bcsc_advanced_evaluation_matrix.png", dpi=180, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(9, 5))
    plt.bar(decile_table["risk_decile_1_highest"], decile_table["event_rate_percent"], color="#2f6f9f")
    plt.axhline(baseline_rate * 100, color="#b42318", linestyle="--", label="baseline")
    plt.gca().invert_xaxis()
    plt.xlabel("Risk decile (1 = highest predicted risk)")
    plt.ylabel("Observed 1-year cancer event rate (%)")
    plt.title("BCSC observed event rate by predicted risk decile")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG / "path1_bcsc_event_rate_by_risk_decile.png", dpi=180, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(9, 5))
    plt.plot(decile_table["cumulative_population_percent"], decile_table["cumulative_event_capture_percent"], marker="o")
    plt.plot([0, 100], [0, 100], linestyle="--", color="#888888", label="random")
    plt.xlabel("Cumulative population selected (%)")
    plt.ylabel("Cumulative cancer events captured (%)")
    plt.title("BCSC cumulative event capture by predicted risk")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG / "path1_bcsc_cumulative_event_capture.png", dpi=180, bbox_inches="tight")
    plt.close()
    return table


def load_metabric():
    patient = pd.read_csv(METABRIC_DATA_DIR / "data_clinical_patient.txt", sep="\t", comment="#")
    sample = pd.read_csv(METABRIC_DATA_DIR / "data_clinical_sample.txt", sep="\t", comment="#")
    clinical = patient.merge(sample, on="PATIENT_ID", how="inner", suffixes=("_PATIENT", "_SAMPLE")).set_index("SAMPLE_ID", drop=False)
    expr_raw = pd.read_csv(METABRIC_DATA_DIR / "data_mrna_illumina_microarray_zscores_ref_diploid_samples.txt", sep="\t")
    expr_raw = expr_raw.dropna(subset=["Hugo_Symbol"]).copy()
    expr_raw["Hugo_Symbol"] = expr_raw["Hugo_Symbol"].astype(str)
    expr_raw = expr_raw.groupby("Hugo_Symbol", sort=False).mean(numeric_only=True)
    expression = expr_raw.T.astype("float32")
    expression.index.name = "SAMPLE_ID"
    shared = clinical.index.intersection(expression.index)
    clinical = clinical.loc[shared].copy()
    expression = expression.loc[shared].copy()
    labels = pd.DataFrame(index=clinical.index)
    labels["os_months"] = pd.to_numeric(clinical["OS_MONTHS"], errors="coerce")
    labels["os_event"] = clinical["OS_STATUS"].map({"1:DECEASED": 1.0, "0:LIVING": 0.0})
    labels["os_5yr_event"] = horizon_label(labels["os_event"], labels["os_months"], 60)
    return clinical, expression, labels


def clinical_matrix(clinical: pd.DataFrame) -> pd.DataFrame:
    numeric = [c for c in [
        "AGE_AT_DIAGNOSIS", "LYMPH_NODES_EXAMINED_POSITIVE", "NPI", "GRADE",
        "TUMOR_SIZE", "TUMOR_STAGE", "TMB_NONSYNONYMOUS", "COHORT",
    ] if c in clinical.columns]
    categorical = [c for c in [
        "CELLULARITY", "CHEMOTHERAPY", "ER_IHC", "ER_STATUS", "HER2_SNP6", "HER2_STATUS",
        "HORMONE_THERAPY", "INFERRED_MENOPAUSAL_STATE", "PR_STATUS", "RADIO_THERAPY",
        "HISTOLOGICAL_SUBTYPE", "BREAST_SURGERY", "CANCER_TYPE_DETAILED", "THREEGENE", "INTCLUST",
    ] if c in clinical.columns]
    return pd.concat(
        [
            clinical[numeric].apply(pd.to_numeric, errors="coerce"),
            pd.get_dummies(clinical[categorical].fillna("Missing").astype(str)),
        ],
        axis=1,
    )


def rank_genes(expression: pd.DataFrame, train_idx: pd.Index, y_train: pd.Series) -> list[str]:
    X = SimpleImputer(strategy="median").fit_transform(expression.loc[train_idx])
    scores, _ = f_classif(X, y_train)
    scores = np.nan_to_num(scores, nan=-np.inf, posinf=np.inf, neginf=-np.inf)
    return expression.columns[np.argsort(scores)[::-1]].tolist()


def scale_parts(train_df: pd.DataFrame, valid_df: pd.DataFrame, hold_df: pd.DataFrame):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    train_arr = imputer.fit_transform(train_df)
    valid_arr = imputer.transform(valid_df.reindex(columns=train_df.columns, fill_value=0))
    hold_arr = imputer.transform(hold_df.reindex(columns=train_df.columns, fill_value=0))
    return scaler.fit_transform(train_arr), scaler.transform(valid_arr), scaler.transform(hold_arr)


def build_integrated(clin_mat, expression, train_idx, valid_idx, hold_idx, ranked_genes, k_genes):
    genes = ranked_genes[:k_genes]
    g_train, g_valid, g_hold = scale_parts(expression.loc[train_idx, genes], expression.loc[valid_idx, genes], expression.loc[hold_idx, genes])
    c_train, c_valid, c_hold = scale_parts(clin_mat.loc[train_idx], clin_mat.loc[valid_idx], clin_mat.loc[hold_idx])
    names = genes + clin_mat.columns.tolist()
    return np.hstack([g_train, c_train]), np.hstack([g_valid, c_valid]), np.hstack([g_hold, c_hold]), names


def survival_candidates(pos_weight: float):
    candidates = [
        ("logreg_balanced", LogisticRegression(C=0.05, max_iter=10000, solver="liblinear", class_weight="balanced", random_state=RANDOM_STATE), False),
        ("extra_trees_balanced", ExtraTreesClassifier(n_estimators=900, min_samples_leaf=2, max_features="sqrt", class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE), False),
        ("hist_gradient_boosting", HistGradientBoostingClassifier(max_iter=260, learning_rate=0.025, max_leaf_nodes=15, l2_regularization=0.05, random_state=RANDOM_STATE), False),
    ]
    if XGBClassifier is not None:
        candidates.append(("xgboost_scale_pos_weight", XGBClassifier(n_estimators=520, max_depth=2, learning_rate=0.025, subsample=0.9, colsample_bytree=0.75, reg_lambda=2.0, scale_pos_weight=pos_weight, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1), False))
        candidates.append(("xgboost_smote", XGBClassifier(n_estimators=420, max_depth=2, learning_rate=0.025, subsample=0.9, colsample_bytree=0.75, reg_lambda=2.0, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1), True))
    if LGBMClassifier is not None:
        candidates.append(("lightgbm_balanced", LGBMClassifier(n_estimators=520, learning_rate=0.025, num_leaves=15, min_child_samples=12, subsample=0.9, colsample_bytree=0.75, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, verbose=-1), False))
        candidates.append(("lightgbm_smote", LGBMClassifier(n_estimators=420, learning_rate=0.025, num_leaves=15, min_child_samples=12, subsample=0.9, colsample_bytree=0.75, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1), True))
    return candidates


def run_metabric_benchmark(target_col: str = "os_5yr_event", label: str = "5yr") -> pd.DataFrame:
    print(f"Running Path 2 METABRIC published-style survival benchmark: {target_col}...")
    clinical, expression, labels = load_metabric()
    clin_mat = clinical_matrix(clinical)
    y = labels[target_col].dropna().astype(int)
    trainval_idx, hold_idx = train_test_split(y.index, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
    train_idx, valid_idx = train_test_split(pd.Index(trainval_idx), test_size=0.25, stratify=y.loc[trainval_idx], random_state=43)
    train_idx, valid_idx, hold_idx = pd.Index(train_idx), pd.Index(valid_idx), pd.Index(hold_idx)
    ranked = rank_genes(expression, train_idx, y.loc[train_idx])
    pos_weight = (y.loc[train_idx].eq(0).sum() / max(y.loc[train_idx].eq(1).sum(), 1))
    rows = []
    best = {"validation_auc": -1}
    best_model = None
    best_payload = None
    for k in [50, 100, 300, 800]:
        X_train, X_valid, X_hold, feature_names = build_integrated(clin_mat, expression, train_idx, valid_idx, hold_idx, ranked, k)
        y_train, y_valid, y_hold = y.loc[train_idx].values, y.loc[valid_idx].values, y.loc[hold_idx].values
        for name, model, use_smote in survival_candidates(pos_weight):
            if use_smote and SMOTE is None:
                continue
            print("  METABRIC model:", name, "k=", k, "smote=", use_smote)
            X_fit, y_fit = X_train, y_train
            if use_smote:
                X_fit, y_fit = SMOTE(random_state=RANDOM_STATE, k_neighbors=5).fit_resample(X_train, y_train)
            model.fit(X_fit, y_fit)
            valid_scores = model.predict_proba(X_valid)[:, 1]
            th_ba, _ = choose_threshold(y_valid, valid_scores, None, "balanced_accuracy")
            th_f1, _ = choose_threshold(y_valid, valid_scores, None, "f1")
            validation_auc = roc_auc_score(y_valid, valid_scores)
            validation_ap = average_precision_score(y_valid, valid_scores)
            hold_scores = model.predict_proba(X_hold)[:, 1]
            for mode, th in [("balanced_accuracy_threshold", th_ba), ("f1_threshold", th_f1), ("fixed_0.5", 0.5)]:
                metrics = full_binary_metrics(y_hold, hold_scores, th)
                rows.append({
                    "model": name,
                    "k_genes": k,
                    "use_smote_train_only": bool(use_smote),
                    "validation_auc": validation_auc,
                    "validation_average_precision": validation_ap,
                    "threshold_mode": mode,
                    **{f"holdout_{key}": value for key, value in metrics.items()},
                    "samples": int(len(y)),
                    "events": int(y.sum()),
                })
            if validation_auc > best["validation_auc"]:
                best = {"validation_auc": validation_auc, "model": name, "k_genes": k, "use_smote": bool(use_smote)}
                best_model = model
                best_payload = {"feature_names": feature_names, "threshold_ba": th_ba, "threshold_f1": th_f1}

    table = pd.DataFrame(rows).sort_values(["holdout_roc_auc", "holdout_balanced_accuracy"], ascending=False)
    table.to_csv(BENCH / f"path2_metabric_published_style_survival_benchmark_{label}.csv", index=False)
    if best_model is not None:
        joblib.dump({"model": best_model, **best, **best_payload}, MODELS / f"path2_metabric_published_style_best_model_{label}.joblib")
    save_json(table.iloc[0].to_dict(), BENCH / f"path2_metabric_published_style_best_metrics_{label}.json")

    top = table.groupby(["model", "k_genes"], as_index=False).first().sort_values("holdout_roc_auc", ascending=False).head(10)
    plt.figure(figsize=(12, 5))
    labels_x = top["model"] + "_k" + top["k_genes"].astype(str)
    x = np.arange(len(top))
    width = 0.18
    for j, metric in enumerate(["holdout_roc_auc", "holdout_accuracy", "holdout_sensitivity_recall", "holdout_specificity"]):
        plt.bar(x + (j - 1.5) * width, top[metric], width, label=metric.replace("holdout_", ""))
    plt.xticks(x, labels_x, rotation=35, ha="right")
    plt.ylim(0, 1)
    plt.title(f"METABRIC Published-style Survival Benchmark ({label})")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG / f"path2_metabric_published_style_survival_benchmark_{label}.png", dpi=180, bbox_inches="tight")
    plt.close()
    return table


def main():
    bcsc = run_bcsc_benchmark()
    met_5yr = run_metabric_benchmark("os_5yr_event", "5yr")
    met_os = run_metabric_benchmark("os_event", "overall_status")
    summary = pd.DataFrame([
        {
            "path": "Path 1 - BCSC",
            "benchmark": "Advanced BCSC future-risk model",
            "best_model": bcsc.iloc[0]["model"],
            "metric": "Validation ROC-AUC",
            "result": round(float(bcsc.iloc[0]["roc_auc"]), 3),
            "accuracy": round(float(bcsc.iloc[0]["accuracy"]), 3),
            "sensitivity": round(float(bcsc.iloc[0]["sensitivity_recall"]), 3),
            "specificity": round(float(bcsc.iloc[0]["specificity"]), 3),
            "f1": round(float(bcsc.iloc[0]["f1"]), 3),
        },
        {
            "path": "Path 2 - METABRIC",
            "benchmark": "Published-style clinical + transcriptomic 5-year survival",
            "best_model": met_5yr.iloc[0]["model"],
            "metric": "Holdout ROC-AUC",
            "result": round(float(met_5yr.iloc[0]["holdout_roc_auc"]), 3),
            "accuracy": round(float(met_5yr.iloc[0]["holdout_accuracy"]), 3),
            "sensitivity": round(float(met_5yr.iloc[0]["holdout_sensitivity_recall"]), 3),
            "specificity": round(float(met_5yr.iloc[0]["holdout_specificity"]), 3),
            "f1": round(float(met_5yr.iloc[0]["holdout_f1"]), 3),
        },
        {
            "path": "Path 2 - METABRIC",
            "benchmark": "Published-style clinical + transcriptomic overall survival status",
            "best_model": met_os.iloc[0]["model"],
            "metric": "Holdout ROC-AUC",
            "result": round(float(met_os.iloc[0]["holdout_roc_auc"]), 3),
            "accuracy": round(float(met_os.iloc[0]["holdout_accuracy"]), 3),
            "sensitivity": round(float(met_os.iloc[0]["holdout_sensitivity_recall"]), 3),
            "specificity": round(float(met_os.iloc[0]["holdout_specificity"]), 3),
            "f1": round(float(met_os.iloc[0]["holdout_f1"]), 3),
        },
    ])
    summary.to_csv(BENCH / "advanced_benchmark_summary.csv", index=False)
    print("\nAdvanced benchmark summary:")
    print(summary.to_string(index=False))
    print("\nOutputs:", BENCH)


main()


## Hybrid Ensemble Benchmark

This section tests hybrid/ensemble models for BCSC and METABRIC. The ensemble is tuned on validation/blend data and evaluated separately to check whether combining models improves performance.


In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.feature_selection import f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBMClassifier = None

try:
    from imblearn.over_sampling import SMOTE
except Exception:
    SMOTE = None


RANDOM_STATE = 42
ROOT = Path.cwd()
BCSC_DATA_DIR = ROOT / "external_datasets" / "BCSC"
METABRIC_DATA_DIR = ROOT / "external_datasets" / "METABRIC" / "brca_metabric_direct"
OUT = ROOT / "outputs" / "clean_two_path_complete"
HYBRID = OUT / "hybrid_benchmark"
FIG = HYBRID / "figures"
MODELS = HYBRID / "models"
for path in [HYBRID, FIG, MODELS]:
    path.mkdir(parents=True, exist_ok=True)


def save_json(obj: dict, path: Path) -> None:
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def choose_threshold(y_true, scores, sample_weight=None, metric: str = "balanced_accuracy"):
    best_threshold, best_score = 0.5, -1.0
    thresholds = np.unique(np.quantile(scores, np.linspace(0.01, 0.99, 199)))
    for threshold in thresholds:
        pred = (scores >= threshold).astype(int)
        if metric == "f1":
            score = f1_score(y_true, pred, sample_weight=sample_weight, zero_division=0)
        else:
            score = balanced_accuracy_score(y_true, pred, sample_weight=sample_weight)
        if score > best_score:
            best_threshold, best_score = float(threshold), float(score)
    return best_threshold, best_score


def metrics(y_true, scores, threshold, sample_weight=None) -> dict:
    pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, sample_weight=sample_weight, labels=[0, 1]).ravel()
    return {
        "roc_auc": float(roc_auc_score(y_true, scores, sample_weight=sample_weight)),
        "average_precision": float(average_precision_score(y_true, scores, sample_weight=sample_weight)),
        "accuracy": float(accuracy_score(y_true, pred, sample_weight=sample_weight)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred, sample_weight=sample_weight)),
        "sensitivity_recall": float(recall_score(y_true, pred, sample_weight=sample_weight, zero_division=0)),
        "specificity": float(tn / (tn + fp) if (tn + fp) else 0.0),
        "precision": float(precision_score(y_true, pred, sample_weight=sample_weight, zero_division=0)),
        "f1": float(f1_score(y_true, pred, sample_weight=sample_weight, zero_division=0)),
        "threshold": float(threshold),
        "tn": float(tn),
        "fp": float(fp),
        "fn": float(fn),
        "tp": float(tp),
    }


def optimize_blend_weights(y_true, score_matrix, sample_weight=None, n_iter: int = 3000) -> np.ndarray:
    rng = np.random.default_rng(RANDOM_STATE)
    n_models = score_matrix.shape[1]
    best_w = np.ones(n_models) / n_models
    best_auc = roc_auc_score(y_true, score_matrix @ best_w, sample_weight=sample_weight)
    for _ in range(n_iter):
        w = rng.dirichlet(np.ones(n_models))
        auc = roc_auc_score(y_true, score_matrix @ w, sample_weight=sample_weight)
        if auc > best_auc:
            best_auc = auc
            best_w = w
    return best_w


def top_decile_lift(y_true, scores, sample_weight=None) -> dict:
    df = pd.DataFrame({"y": np.asarray(y_true), "score": scores, "w": np.ones(len(scores)) if sample_weight is None else np.asarray(sample_weight)})
    df = df.sort_values("score", ascending=False).reset_index(drop=True)
    total_w = df["w"].sum()
    total_e = (df["y"] * df["w"]).sum()
    baseline = total_e / total_w
    df["cw"] = df["w"].cumsum()
    top = df[df["cw"] <= total_w * 0.1]
    if top.empty:
        top = df.head(max(1, int(len(df) * 0.1)))
    top_w = top["w"].sum()
    top_e = (top["y"] * top["w"]).sum()
    top_rate = top_e / top_w
    return {
        "baseline_event_rate_percent": float(baseline * 100),
        "top_decile_event_rate_percent": float(top_rate * 100),
        "top_decile_lift": float(top_rate / baseline if baseline else np.nan),
        "top_decile_event_capture_percent": float(top_e / total_e * 100 if total_e else 0),
    }


def run_bcsc_hybrid() -> pd.DataFrame:
    print("Running BCSC hybrid benchmark...")
    cols = [
        "menopaus", "agegrp", "density", "race", "hispanic", "bmi",
        "agefirst", "nrelbc", "brstproc", "lastmamm", "surgmeno", "hrt",
        "invasive", "cancer", "training", "count",
    ]
    features = cols[:12]
    bcsc = pd.read_csv(BCSC_DATA_DIR / "bcsc_risk_estimation_dataset_1" / "risk.txt", sep=r"\s+", header=None, names=cols, engine="python")
    for col in features + ["cancer", "training"]:
        bcsc[col] = bcsc[col].astype(str)
    bcsc["count"] = pd.to_numeric(bcsc["count"], errors="coerce").fillna(0).astype(float)
    train_mask = bcsc["training"].eq("1")
    valid_mask = bcsc["training"].eq("0")
    train_df = bcsc.loc[train_mask].copy()
    official_valid = bcsc.loc[valid_mask].copy()
    inner_train_idx, blend_idx = train_test_split(train_df.index, test_size=0.25, stratify=train_df["cancer"].astype(int), random_state=RANDOM_STATE)

    def balanced_weights(y, w):
        pos = float(w[y.eq(1)].sum())
        neg = float(w[y.eq(0)].sum())
        total = pos + neg
        mult = {0: total / (2 * neg), 1: total / (2 * pos)}
        return w * y.map(mult).astype(float)

    onehot = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), features)])
    ordinal = ColumnTransformer([("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=999), features)])
    model_specs = [
        ("logreg", Pipeline([("prep", onehot), ("model", LogisticRegression(max_iter=10000, C=1.0, random_state=RANDOM_STATE))]), "original"),
        ("logreg_balanced_weight", Pipeline([("prep", onehot), ("model", LogisticRegression(max_iter=10000, C=1.0, random_state=RANDOM_STATE))]), "balanced"),
    ]
    if XGBClassifier is not None:
        model_specs.append(("xgboost", Pipeline([("prep", ordinal), ("model", XGBClassifier(n_estimators=450, max_depth=3, learning_rate=0.025, subsample=0.9, colsample_bytree=0.9, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1))]), "original"))
        model_specs.append(("xgboost_balanced_weight", Pipeline([("prep", ordinal), ("model", XGBClassifier(n_estimators=520, max_depth=3, learning_rate=0.02, subsample=0.9, colsample_bytree=0.9, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1))]), "balanced"))
    if LGBMClassifier is not None:
        model_specs.append(("lightgbm", Pipeline([("prep", ordinal), ("model", LGBMClassifier(n_estimators=500, learning_rate=0.025, num_leaves=15, min_child_samples=20, subsample=0.9, colsample_bytree=0.9, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1))]), "original"))
        model_specs.append(("lightgbm_balanced_weight", Pipeline([("prep", ordinal), ("model", LGBMClassifier(n_estimators=620, learning_rate=0.02, num_leaves=15, min_child_samples=20, subsample=0.9, colsample_bytree=0.9, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1))]), "balanced"))

    blend_scores = []
    blend_names = []
    y_blend = train_df.loc[blend_idx, "cancer"].astype(int)
    w_blend = train_df.loc[blend_idx, "count"]
    for name, model, weight_mode in model_specs:
        print("  inner fit:", name)
        y_inner = train_df.loc[inner_train_idx, "cancer"].astype(int)
        w_inner = train_df.loc[inner_train_idx, "count"]
        fit_w = balanced_weights(y_inner, w_inner) if weight_mode == "balanced" else w_inner
        model.fit(train_df.loc[inner_train_idx, features], y_inner, model__sample_weight=fit_w)
        blend_scores.append(model.predict_proba(train_df.loc[blend_idx, features])[:, 1])
        blend_names.append(name)
    blend_matrix = np.vstack(blend_scores).T
    weights = optimize_blend_weights(y_blend, blend_matrix, w_blend, n_iter=4000)

    valid_scores = []
    fitted = {}
    y_train = train_df["cancer"].astype(int)
    w_train = train_df["count"]
    y_valid = official_valid["cancer"].astype(int)
    w_valid = official_valid["count"]
    for name, model, weight_mode in model_specs:
        print("  full fit:", name)
        fit_w = balanced_weights(y_train, w_train) if weight_mode == "balanced" else w_train
        model.fit(train_df[features], y_train, model__sample_weight=fit_w)
        valid_scores.append(model.predict_proba(official_valid[features])[:, 1])
        fitted[name] = model
    valid_matrix = np.vstack(valid_scores).T
    hybrid_scores = valid_matrix @ weights
    th, _ = choose_threshold(y_valid, hybrid_scores, w_valid, "balanced_accuracy")
    rows = []
    for name, scores in zip(blend_names, valid_scores):
        th_i, _ = choose_threshold(y_valid, scores, w_valid, "balanced_accuracy")
        rows.append({"model": name, "type": "base", **metrics(y_valid, scores, th_i, w_valid), **top_decile_lift(y_valid, scores, w_valid)})
    rows.append({"model": "hybrid_weighted_blend", "type": "hybrid", **metrics(y_valid, hybrid_scores, th, w_valid), **top_decile_lift(y_valid, hybrid_scores, w_valid)})
    table = pd.DataFrame(rows).sort_values(["roc_auc", "balanced_accuracy"], ascending=False)
    table.to_csv(HYBRID / "path1_bcsc_hybrid_metrics.csv", index=False)
    save_json({"model_names": blend_names, "weights": {n: float(w) for n, w in zip(blend_names, weights)}, "best": table.iloc[0].to_dict()}, HYBRID / "path1_bcsc_hybrid_summary.json")
    joblib.dump({"models": fitted, "weights": weights, "names": blend_names}, MODELS / "path1_bcsc_hybrid_model.joblib")

    plot_df = table.sort_values("roc_auc", ascending=False)
    plt.figure(figsize=(11, 5))
    x = np.arange(len(plot_df))
    width = 0.18
    for j, metric in enumerate(["roc_auc", "balanced_accuracy", "sensitivity_recall", "top_decile_lift"]):
        values = plot_df[metric]
        if metric == "top_decile_lift":
            values = values / values.max()
        plt.bar(x + (j - 1.5) * width, values, width, label=metric)
    plt.xticks(x, plot_df["model"], rotation=35, ha="right")
    plt.title("BCSC Hybrid Benchmark")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG / "path1_bcsc_hybrid_benchmark.png", dpi=180, bbox_inches="tight")
    plt.close()
    return table


def horizon_label(events: pd.Series, months: pd.Series, horizon: float = 60.0) -> pd.Series:
    y = pd.Series(index=events.index, dtype=float)
    y[(events.eq(1)) & (months <= horizon)] = 1.0
    y[(months >= horizon) & ~((events.eq(1)) & (months <= horizon))] = 0.0
    return y


def load_metabric():
    patient = pd.read_csv(METABRIC_DATA_DIR / "data_clinical_patient.txt", sep="\t", comment="#")
    sample = pd.read_csv(METABRIC_DATA_DIR / "data_clinical_sample.txt", sep="\t", comment="#")
    clinical = patient.merge(sample, on="PATIENT_ID", how="inner", suffixes=("_PATIENT", "_SAMPLE")).set_index("SAMPLE_ID", drop=False)
    expr_raw = pd.read_csv(METABRIC_DATA_DIR / "data_mrna_illumina_microarray_zscores_ref_diploid_samples.txt", sep="\t")
    expr_raw = expr_raw.dropna(subset=["Hugo_Symbol"]).copy()
    expr_raw["Hugo_Symbol"] = expr_raw["Hugo_Symbol"].astype(str)
    expr_raw = expr_raw.groupby("Hugo_Symbol", sort=False).mean(numeric_only=True)
    expression = expr_raw.T.astype("float32")
    expression.index.name = "SAMPLE_ID"
    shared = clinical.index.intersection(expression.index)
    clinical = clinical.loc[shared].copy()
    expression = expression.loc[shared].copy()
    labels = pd.DataFrame(index=clinical.index)
    labels["os_months"] = pd.to_numeric(clinical["OS_MONTHS"], errors="coerce")
    labels["os_event"] = clinical["OS_STATUS"].map({"1:DECEASED": 1.0, "0:LIVING": 0.0})
    labels["os_5yr_event"] = horizon_label(labels["os_event"], labels["os_months"], 60)
    return clinical, expression, labels


def clinical_matrix(clinical):
    numeric = [c for c in ["AGE_AT_DIAGNOSIS", "LYMPH_NODES_EXAMINED_POSITIVE", "NPI", "GRADE", "TUMOR_SIZE", "TUMOR_STAGE", "TMB_NONSYNONYMOUS", "COHORT"] if c in clinical.columns]
    categorical = [c for c in ["CELLULARITY", "CHEMOTHERAPY", "ER_IHC", "ER_STATUS", "HER2_SNP6", "HER2_STATUS", "HORMONE_THERAPY", "INFERRED_MENOPAUSAL_STATE", "PR_STATUS", "RADIO_THERAPY", "HISTOLOGICAL_SUBTYPE", "BREAST_SURGERY", "CANCER_TYPE_DETAILED", "THREEGENE", "INTCLUST"] if c in clinical.columns]
    return pd.concat([clinical[numeric].apply(pd.to_numeric, errors="coerce"), pd.get_dummies(clinical[categorical].fillna("Missing").astype(str))], axis=1)


def rank_genes(expression, train_idx, y_train):
    X = SimpleImputer(strategy="median").fit_transform(expression.loc[train_idx])
    scores, _ = f_classif(X, y_train)
    scores = np.nan_to_num(scores, nan=-np.inf, posinf=np.inf, neginf=-np.inf)
    return expression.columns[np.argsort(scores)[::-1]].tolist()


def scale_parts(train_df, valid_df, hold_df):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    train_arr = imputer.fit_transform(train_df)
    valid_arr = imputer.transform(valid_df.reindex(columns=train_df.columns, fill_value=0))
    hold_arr = imputer.transform(hold_df.reindex(columns=train_df.columns, fill_value=0))
    return scaler.fit_transform(train_arr), scaler.transform(valid_arr), scaler.transform(hold_arr)


def build_integrated(clin_mat, expression, train_idx, valid_idx, hold_idx, ranked_genes, k_genes):
    genes = ranked_genes[:k_genes]
    g_train, g_valid, g_hold = scale_parts(expression.loc[train_idx, genes], expression.loc[valid_idx, genes], expression.loc[hold_idx, genes])
    c_train, c_valid, c_hold = scale_parts(clin_mat.loc[train_idx], clin_mat.loc[valid_idx], clin_mat.loc[hold_idx])
    return np.hstack([g_train, c_train]), np.hstack([g_valid, c_valid]), np.hstack([g_hold, c_hold])


def run_metabric_hybrid() -> pd.DataFrame:
    print("Running METABRIC hybrid benchmark...")
    clinical, expression, labels = load_metabric()
    clin_mat = clinical_matrix(clinical)
    y = labels["os_5yr_event"].dropna().astype(int)
    trainval_idx, hold_idx = train_test_split(y.index, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
    train_idx, valid_idx = train_test_split(pd.Index(trainval_idx), test_size=0.25, stratify=y.loc[trainval_idx], random_state=43)
    train_idx, valid_idx, hold_idx = pd.Index(train_idx), pd.Index(valid_idx), pd.Index(hold_idx)
    ranked = rank_genes(expression, train_idx, y.loc[train_idx])
    X_train, X_valid, X_hold = build_integrated(clin_mat, expression, train_idx, valid_idx, hold_idx, ranked, 800)
    y_train, y_valid, y_hold = y.loc[train_idx].values, y.loc[valid_idx].values, y.loc[hold_idx].values
    pos_weight = (y.loc[train_idx].eq(0).sum() / max(y.loc[train_idx].eq(1).sum(), 1))
    specs = [
        ("logreg", LogisticRegression(C=0.05, max_iter=10000, solver="liblinear", class_weight="balanced", random_state=RANDOM_STATE), False),
        ("extra_trees", ExtraTreesClassifier(n_estimators=900, min_samples_leaf=2, max_features="sqrt", class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE), False),
        ("hist_gradient", HistGradientBoostingClassifier(max_iter=280, learning_rate=0.025, max_leaf_nodes=15, l2_regularization=0.05, random_state=RANDOM_STATE), False),
    ]
    if XGBClassifier is not None:
        specs.append(("xgboost", XGBClassifier(n_estimators=520, max_depth=2, learning_rate=0.025, subsample=0.9, colsample_bytree=0.75, reg_lambda=2.0, scale_pos_weight=pos_weight, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1), False))
        specs.append(("xgboost_smote", XGBClassifier(n_estimators=420, max_depth=2, learning_rate=0.025, subsample=0.9, colsample_bytree=0.75, reg_lambda=2.0, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1), True))
    if LGBMClassifier is not None:
        specs.append(("lightgbm", LGBMClassifier(n_estimators=540, learning_rate=0.025, num_leaves=15, min_child_samples=12, subsample=0.9, colsample_bytree=0.75, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, verbose=-1), False))
        specs.append(("lightgbm_smote", LGBMClassifier(n_estimators=440, learning_rate=0.025, num_leaves=15, min_child_samples=12, subsample=0.9, colsample_bytree=0.75, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1), True))
    valid_scores, hold_scores, names, fitted = [], [], [], {}
    for name, model, use_smote in specs:
        print("  fit:", name)
        X_fit, y_fit = X_train, y_train
        if use_smote and SMOTE is not None:
            X_fit, y_fit = SMOTE(random_state=RANDOM_STATE, k_neighbors=5).fit_resample(X_train, y_train)
        model.fit(X_fit, y_fit)
        valid_scores.append(model.predict_proba(X_valid)[:, 1])
        hold_scores.append(model.predict_proba(X_hold)[:, 1])
        names.append(name)
        fitted[name] = model
    valid_matrix = np.vstack(valid_scores).T
    hold_matrix = np.vstack(hold_scores).T
    weights = optimize_blend_weights(y_valid, valid_matrix, None, n_iter=5000)
    hybrid_valid = valid_matrix @ weights
    hybrid_hold = hold_matrix @ weights
    threshold, _ = choose_threshold(y_valid, hybrid_valid, None, "balanced_accuracy")
    rows = []
    for name, v, h in zip(names, valid_scores, hold_scores):
        th, _ = choose_threshold(y_valid, v, None, "balanced_accuracy")
        rows.append({"model": name, "type": "base", "validation_auc": roc_auc_score(y_valid, v), **{f"holdout_{k}": val for k, val in metrics(y_hold, h, th).items()}})
    rows.append({"model": "hybrid_weighted_blend", "type": "hybrid", "validation_auc": roc_auc_score(y_valid, hybrid_valid), **{f"holdout_{k}": val for k, val in metrics(y_hold, hybrid_hold, threshold).items()}})
    table = pd.DataFrame(rows).sort_values(["holdout_roc_auc", "holdout_balanced_accuracy"], ascending=False)
    table.to_csv(HYBRID / "path2_metabric_hybrid_survival_metrics.csv", index=False)
    save_json({"model_names": names, "weights": {n: float(w) for n, w in zip(names, weights)}, "best": table.iloc[0].to_dict()}, HYBRID / "path2_metabric_hybrid_summary.json")
    joblib.dump({"models": fitted, "weights": weights, "names": names}, MODELS / "path2_metabric_hybrid_model.joblib")

    plt.figure(figsize=(11, 5))
    plot_df = table.copy()
    x = np.arange(len(plot_df))
    width = 0.18
    for j, metric in enumerate(["holdout_roc_auc", "holdout_balanced_accuracy", "holdout_sensitivity_recall", "holdout_specificity"]):
        plt.bar(x + (j - 1.5) * width, plot_df[metric], width, label=metric.replace("holdout_", ""))
    plt.xticks(x, plot_df["model"], rotation=35, ha="right")
    plt.ylim(0, 1)
    plt.title("METABRIC Hybrid 5-year Survival Benchmark")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG / "path2_metabric_hybrid_survival_benchmark.png", dpi=180, bbox_inches="tight")
    plt.close()
    return table


def main():
    bcsc = run_bcsc_hybrid()
    met = run_metabric_hybrid()
    summary = pd.DataFrame([
        {
            "path": "Path 1 - BCSC",
            "benchmark": "Hybrid future-risk ensemble",
            "best_model": bcsc.iloc[0]["model"],
            "auc": round(float(bcsc.iloc[0]["roc_auc"]), 3),
            "accuracy": round(float(bcsc.iloc[0]["accuracy"]), 3),
            "sensitivity": round(float(bcsc.iloc[0]["sensitivity_recall"]), 3),
            "specificity": round(float(bcsc.iloc[0]["specificity"]), 3),
            "f1": round(float(bcsc.iloc[0]["f1"]), 3),
            "top_decile_lift": round(float(bcsc.iloc[0]["top_decile_lift"]), 2),
        },
        {
            "path": "Path 2 - METABRIC",
            "benchmark": "Hybrid clinical + transcriptomic 5-year survival ensemble",
            "best_model": met.iloc[0]["model"],
            "auc": round(float(met.iloc[0]["holdout_roc_auc"]), 3),
            "accuracy": round(float(met.iloc[0]["holdout_accuracy"]), 3),
            "sensitivity": round(float(met.iloc[0]["holdout_sensitivity_recall"]), 3),
            "specificity": round(float(met.iloc[0]["holdout_specificity"]), 3),
            "f1": round(float(met.iloc[0]["holdout_f1"]), 3),
            "top_decile_lift": "",
        },
    ])
    summary.to_csv(HYBRID / "hybrid_benchmark_summary.csv", index=False)
    print(summary.to_string(index=False))


main()


## Survival Analysis, Repeated CV, and Paper-Style Benchmark

This section adds Cox survival analysis, clean repeated cross-validation, and a clearly labeled paper-style benchmark using global feature selection for comparison with published-style reporting.


In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from sklearn.feature_selection import f_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBMClassifier = None

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

try:
    from imblearn.over_sampling import SMOTE
except Exception:
    SMOTE = None


RANDOM_STATE = 42
ROOT = Path.cwd()
METABRIC_DATA_DIR = ROOT / "external_datasets" / "METABRIC" / "brca_metabric_direct"
OUT = ROOT / "outputs" / "clean_two_path_complete"
BENCH = OUT / "survival_cv_paperstyle_benchmark"
FIG = BENCH / "figures"
for path in [BENCH, FIG]:
    path.mkdir(parents=True, exist_ok=True)


def save_json(obj: dict, path: Path) -> None:
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def horizon_label(events: pd.Series, months: pd.Series, horizon: float = 60.0) -> pd.Series:
    y = pd.Series(index=events.index, dtype=float)
    y[(events.eq(1)) & (months <= horizon)] = 1.0
    y[(months >= horizon) & ~((events.eq(1)) & (months <= horizon))] = 0.0
    return y


def load_metabric():
    patient = pd.read_csv(METABRIC_DATA_DIR / "data_clinical_patient.txt", sep="\t", comment="#")
    sample = pd.read_csv(METABRIC_DATA_DIR / "data_clinical_sample.txt", sep="\t", comment="#")
    clinical = patient.merge(sample, on="PATIENT_ID", how="inner", suffixes=("_PATIENT", "_SAMPLE")).set_index("SAMPLE_ID", drop=False)
    expr_raw = pd.read_csv(METABRIC_DATA_DIR / "data_mrna_illumina_microarray_zscores_ref_diploid_samples.txt", sep="\t")
    expr_raw = expr_raw.dropna(subset=["Hugo_Symbol"]).copy()
    expr_raw["Hugo_Symbol"] = expr_raw["Hugo_Symbol"].astype(str)
    expr_raw = expr_raw.groupby("Hugo_Symbol", sort=False).mean(numeric_only=True)
    expression = expr_raw.T.astype("float32")
    expression.index.name = "SAMPLE_ID"
    shared = clinical.index.intersection(expression.index)
    clinical = clinical.loc[shared].copy()
    expression = expression.loc[shared].copy()

    labels = pd.DataFrame(index=clinical.index)
    labels["os_months"] = pd.to_numeric(clinical["OS_MONTHS"], errors="coerce")
    labels["os_event"] = clinical["OS_STATUS"].map({"1:DECEASED": 1.0, "0:LIVING": 0.0})
    labels["os_5yr_event"] = horizon_label(labels["os_event"], labels["os_months"], 60)
    return clinical, expression, labels


def make_clinical_matrix(clinical: pd.DataFrame) -> pd.DataFrame:
    numeric = [c for c in [
        "AGE_AT_DIAGNOSIS", "LYMPH_NODES_EXAMINED_POSITIVE", "NPI", "GRADE",
        "TUMOR_SIZE", "TUMOR_STAGE", "TMB_NONSYNONYMOUS", "COHORT",
    ] if c in clinical.columns]
    categorical = [c for c in [
        "CELLULARITY", "CHEMOTHERAPY", "ER_IHC", "ER_STATUS", "HER2_SNP6", "HER2_STATUS",
        "HORMONE_THERAPY", "INFERRED_MENOPAUSAL_STATE", "PR_STATUS", "RADIO_THERAPY",
        "HISTOLOGICAL_SUBTYPE", "BREAST_SURGERY", "CANCER_TYPE_DETAILED", "THREEGENE", "INTCLUST",
    ] if c in clinical.columns]
    return pd.concat(
        [
            clinical[numeric].apply(pd.to_numeric, errors="coerce"),
            pd.get_dummies(clinical[categorical].fillna("Missing").astype(str)),
        ],
        axis=1,
    )


def rank_genes(expression: pd.DataFrame, idx: pd.Index, y: pd.Series) -> list[str]:
    X = SimpleImputer(strategy="median").fit_transform(expression.loc[idx])
    scores, _ = f_classif(X, y.loc[idx])
    scores = np.nan_to_num(scores, nan=-np.inf, posinf=np.inf, neginf=-np.inf)
    return expression.columns[np.argsort(scores)[::-1]].tolist()


def build_matrix(clin_mat, expression, train_idx, test_idx, ranked_genes, k_genes):
    genes = ranked_genes[:k_genes]
    train_df = pd.concat([expression.loc[train_idx, genes], clin_mat.loc[train_idx]], axis=1)
    test_df = pd.concat([expression.loc[test_idx, genes], clin_mat.loc[test_idx]], axis=1)
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    X_train = scaler.fit_transform(imputer.fit_transform(train_df))
    X_test = scaler.transform(imputer.transform(test_df.reindex(columns=train_df.columns, fill_value=0)))
    return X_train, X_test, train_df.columns.tolist()


def threshold_metrics(y_true, scores):
    thresholds = np.unique(np.quantile(scores, np.linspace(0.05, 0.95, 91)))
    best_t, best_ba = 0.5, -1
    for t in thresholds:
        pred = (scores >= t).astype(int)
        ba = balanced_accuracy_score(y_true, pred)
        if ba > best_ba:
            best_t, best_ba = float(t), float(ba)
    pred = (scores >= best_t).astype(int)
    return {
        "threshold": best_t,
        "roc_auc": roc_auc_score(y_true, scores),
        "average_precision": average_precision_score(y_true, scores),
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "sensitivity": recall_score(y_true, pred, zero_division=0),
        "specificity": recall_score(1 - y_true, 1 - pred, zero_division=0),
        "precision": precision_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
    }


def make_model():
    if LGBMClassifier is not None:
        return LGBMClassifier(
            n_estimators=420,
            learning_rate=0.025,
            num_leaves=15,
            min_child_samples=12,
            subsample=0.9,
            colsample_bytree=0.75,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        )
    if XGBClassifier is not None:
        return XGBClassifier(
            n_estimators=420,
            max_depth=2,
            learning_rate=0.025,
            subsample=0.9,
            colsample_bytree=0.75,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    raise RuntimeError("Need LightGBM or XGBoost for this benchmark")


def run_cox_survival(clinical, expression, labels) -> dict:
    usable = labels[["os_months", "os_event"]].dropna()
    train_idx, hold_idx = train_test_split(usable.index, test_size=0.2, stratify=usable["os_event"].astype(int), random_state=RANDOM_STATE)
    train_idx, hold_idx = pd.Index(train_idx), pd.Index(hold_idx)
    clin_mat = make_clinical_matrix(clinical)
    y_binary_for_genes = labels.loc[train_idx, "os_event"].astype(int)
    ranked = rank_genes(expression, train_idx, y_binary_for_genes)
    genes = ranked[:30]
    cox_df = pd.concat([labels[["os_months", "os_event"]], expression[genes], clin_mat], axis=1).loc[usable.index]
    cox_df = cox_df.replace([np.inf, -np.inf], np.nan)
    keep_cols = ["os_months", "os_event"] + genes + clin_mat.columns.tolist()
    cox_df = cox_df[keep_cols]
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    X_train = pd.DataFrame(scaler.fit_transform(imputer.fit_transform(cox_df.loc[train_idx].drop(columns=["os_months", "os_event"]))), index=train_idx, columns=keep_cols[2:])
    X_hold = pd.DataFrame(scaler.transform(imputer.transform(cox_df.loc[hold_idx].drop(columns=["os_months", "os_event"]))), index=hold_idx, columns=keep_cols[2:])
    train_cox = pd.concat([cox_df.loc[train_idx, ["os_months", "os_event"]], X_train], axis=1)
    hold_cox = pd.concat([cox_df.loc[hold_idx, ["os_months", "os_event"]], X_hold], axis=1)
    cph = CoxPHFitter(penalizer=0.2, l1_ratio=0.05)
    cph.fit(train_cox, duration_col="os_months", event_col="os_event")
    risk = cph.predict_partial_hazard(hold_cox).values.ravel()
    c_index = concordance_index(hold_cox["os_months"], -risk, hold_cox["os_event"])
    out = {
        "benchmark": "Cox clinical + transcriptomic survival analysis",
        "metric": "Holdout C-index",
        "result": float(c_index),
        "samples": int(len(usable)),
        "events": int(usable["os_event"].sum()),
        "k_genes": len(genes),
        "penalizer": 0.2,
    }
    save_json(out, BENCH / "cox_survival_cindex_metrics.json")
    return out


def run_repeated_cv(clinical, expression, labels, target_col: str, label: str, paper_style: bool = False) -> pd.DataFrame:
    y = labels[target_col].dropna().astype(int)
    clin_mat = make_clinical_matrix(clinical).loc[y.index]
    expression_y = expression.loc[y.index]
    rows = []
    splitter = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE) if not paper_style else StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    global_ranked = rank_genes(expression_y, y.index, y) if paper_style else None
    k_genes = 800 if paper_style else 300
    for fold, (train_pos, test_pos) in enumerate(splitter.split(np.zeros(len(y)), y.values), 1):
        train_idx = y.index[train_pos]
        test_idx = y.index[test_pos]
        ranked = global_ranked if paper_style else rank_genes(expression_y, train_idx, y)
        X_train, X_test, _ = build_matrix(clin_mat, expression_y, train_idx, test_idx, ranked, k_genes)
        y_train, y_test = y.loc[train_idx].values, y.loc[test_idx].values
        X_fit, y_fit = X_train, y_train
        if paper_style and SMOTE is not None:
            X_fit, y_fit = SMOTE(random_state=RANDOM_STATE, k_neighbors=5).fit_resample(X_train, y_train)
        model = make_model()
        model.fit(X_fit, y_fit)
        scores = model.predict_proba(X_test)[:, 1]
        row = {
            "fold": fold,
            "target": target_col,
            "style": "paper_style_global_feature_selection" if paper_style else "clean_repeated_cv",
            "k_genes": k_genes,
            **threshold_metrics(y_test, scores),
        }
        rows.append(row)
    table = pd.DataFrame(rows)
    table.to_csv(BENCH / f"{label}_fold_metrics.csv", index=False)
    summary = table.drop(columns=["fold"]).groupby(["target", "style", "k_genes"], as_index=False).agg(["mean", "std"])
    summary.columns = ["_".join([str(x) for x in col if x]) for col in summary.columns]
    summary = summary.reset_index()
    summary.to_csv(BENCH / f"{label}_summary.csv", index=False)
    return table


def main() -> None:
    print("Loading METABRIC...")
    clinical, expression, labels = load_metabric()
    print("Running Cox survival analysis...")
    cox = run_cox_survival(clinical, expression, labels)
    print("Running clean repeated CV...")
    clean_5yr = run_repeated_cv(clinical, expression, labels, "os_5yr_event", "clean_repeated_cv_5yr", paper_style=False)
    print("Running paper-style CV for 5-year survival...")
    paper_5yr = run_repeated_cv(clinical, expression, labels, "os_5yr_event", "paper_style_cv_5yr", paper_style=True)
    print("Running paper-style CV for overall survival status...")
    paper_os = run_repeated_cv(clinical, expression, labels, "os_event", "paper_style_cv_overall_status", paper_style=True)

    summary_rows = []
    for name, table in [
        ("Clean repeated CV 5-year survival", clean_5yr),
        ("Paper-style CV 5-year survival", paper_5yr),
        ("Paper-style CV overall survival status", paper_os),
    ]:
        summary_rows.append({
            "benchmark": name,
            "auc_mean": table["roc_auc"].mean(),
            "auc_std": table["roc_auc"].std(),
            "accuracy_mean": table["accuracy"].mean(),
            "sensitivity_mean": table["sensitivity"].mean(),
            "specificity_mean": table["specificity"].mean(),
            "f1_mean": table["f1"].mean(),
        })
    summary_rows.append({
        "benchmark": cox["benchmark"],
        "auc_mean": cox["result"],
        "auc_std": np.nan,
        "accuracy_mean": np.nan,
        "sensitivity_mean": np.nan,
        "specificity_mean": np.nan,
        "f1_mean": np.nan,
    })
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(BENCH / "survival_cv_paperstyle_summary.csv", index=False)

    print(summary.to_string(index=False))
    print("Outputs:", BENCH)


main()


## Feature Engineering and Gene Signature Benchmark

This section tests interaction features for BCSC and biologically motivated gene-signature scores for METABRIC. The goal is to check whether domain feature engineering improves the clean benchmark.


In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")

try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBMClassifier = None

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None


RANDOM_STATE = 42
ROOT = Path.cwd()
BCSC_DATA_DIR = ROOT / "external_datasets" / "BCSC"
METABRIC_DATA_DIR = ROOT / "external_datasets" / "METABRIC" / "brca_metabric_direct"
OUT = ROOT / "outputs" / "clean_two_path_complete"
BENCH = OUT / "feature_engineered_benchmark"
MODELS = BENCH / "models"
for path in [BENCH, MODELS]:
    path.mkdir(parents=True, exist_ok=True)


def save_json(obj: dict, path: Path) -> None:
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def choose_threshold(y_true, scores, sample_weight=None):
    best_t, best_ba = 0.5, -1.0
    for t in np.unique(np.quantile(scores, np.linspace(0.01, 0.99, 199))):
        pred = (scores >= t).astype(int)
        ba = balanced_accuracy_score(y_true, pred, sample_weight=sample_weight)
        if ba > best_ba:
            best_t, best_ba = float(t), float(ba)
    return best_t


def metrics(y_true, scores, threshold, sample_weight=None) -> dict:
    pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, sample_weight=sample_weight, labels=[0, 1]).ravel()
    return {
        "roc_auc": float(roc_auc_score(y_true, scores, sample_weight=sample_weight)),
        "average_precision": float(average_precision_score(y_true, scores, sample_weight=sample_weight)),
        "accuracy": float(accuracy_score(y_true, pred, sample_weight=sample_weight)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred, sample_weight=sample_weight)),
        "sensitivity": float(recall_score(y_true, pred, sample_weight=sample_weight, zero_division=0)),
        "specificity": float(tn / (tn + fp) if (tn + fp) else 0.0),
        "precision": float(precision_score(y_true, pred, sample_weight=sample_weight, zero_division=0)),
        "f1": float(f1_score(y_true, pred, sample_weight=sample_weight, zero_division=0)),
        "threshold": float(threshold),
    }


def top_decile_lift(y_true, scores, sample_weight=None) -> dict:
    df = pd.DataFrame({"y": np.asarray(y_true), "score": scores, "w": np.ones(len(scores)) if sample_weight is None else np.asarray(sample_weight)})
    df = df.sort_values("score", ascending=False).reset_index(drop=True)
    total_w = float(df["w"].sum())
    total_e = float((df["y"] * df["w"]).sum())
    baseline = total_e / total_w
    df["cw"] = df["w"].cumsum()
    top = df[df["cw"] <= 0.1 * total_w]
    if top.empty:
        top = df.head(max(1, int(len(df) * 0.1)))
    top_w = float(top["w"].sum())
    top_e = float((top["y"] * top["w"]).sum())
    top_rate = top_e / top_w
    return {
        "baseline_event_rate_percent": baseline * 100,
        "top_decile_event_rate_percent": top_rate * 100,
        "top_decile_lift": top_rate / baseline if baseline else np.nan,
        "top_decile_event_capture_percent": top_e / total_e * 100 if total_e else 0.0,
    }


def balanced_weights(y: pd.Series, w: pd.Series) -> pd.Series:
    pos = float(w[y.eq(1)].sum())
    neg = float(w[y.eq(0)].sum())
    total = pos + neg
    mult = {0: total / (2 * neg), 1: total / (2 * pos)}
    return w * y.map(mult).astype(float)


def add_bcsc_interactions(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    pairs = [
        ("agegrp", "density"),
        ("agegrp", "nrelbc"),
        ("agegrp", "bmi"),
        ("density", "nrelbc"),
        ("menopaus", "hrt"),
        ("menopaus", "surgmeno"),
        ("lastmamm", "brstproc"),
        ("agefirst", "nrelbc"),
        ("density", "lastmamm"),
    ]
    for a, b in pairs:
        out[f"{a}_x_{b}"] = out[a].astype(str) + "__" + out[b].astype(str)
    return out


def run_bcsc_feature_engineering() -> pd.DataFrame:
    print("Running BCSC interaction feature benchmark...")
    cols = [
        "menopaus", "agegrp", "density", "race", "hispanic", "bmi",
        "agefirst", "nrelbc", "brstproc", "lastmamm", "surgmeno", "hrt",
        "invasive", "cancer", "training", "count",
    ]
    base_features = cols[:12]
    bcsc = pd.read_csv(BCSC_DATA_DIR / "bcsc_risk_estimation_dataset_1" / "risk.txt", sep=r"\s+", header=None, names=cols, engine="python")
    for col in base_features + ["cancer", "training"]:
        bcsc[col] = bcsc[col].astype(str)
    bcsc["count"] = pd.to_numeric(bcsc["count"], errors="coerce").fillna(0).astype(float)
    bcsc = add_bcsc_interactions(bcsc)
    features = [c for c in bcsc.columns if c in base_features or "_x_" in c]
    train_mask, valid_mask = bcsc["training"].eq("1"), bcsc["training"].eq("0")
    X_train, y_train, w_train = bcsc.loc[train_mask, features], bcsc.loc[train_mask, "cancer"].astype(int), bcsc.loc[train_mask, "count"]
    X_valid, y_valid, w_valid = bcsc.loc[valid_mask, features], bcsc.loc[valid_mask, "cancer"].astype(int), bcsc.loc[valid_mask, "count"]
    w_bal = balanced_weights(y_train, w_train)

    onehot = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=2), features)])
    ordinal = ColumnTransformer([("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=999), features)])
    candidates = [
        ("interaction_logreg_balanced_weight", Pipeline([("prep", onehot), ("model", LogisticRegression(max_iter=10000, C=0.5, random_state=RANDOM_STATE))]), w_bal),
    ]
    if LGBMClassifier is not None:
        candidates.append((
            "interaction_lightgbm_balanced_weight",
            Pipeline([("prep", ordinal), ("model", LGBMClassifier(n_estimators=900, learning_rate=0.015, num_leaves=23, min_child_samples=12, subsample=0.9, colsample_bytree=0.9, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1))]),
            w_bal,
        ))
    if XGBClassifier is not None:
        candidates.append((
            "interaction_xgboost_balanced_weight",
            Pipeline([("prep", ordinal), ("model", XGBClassifier(n_estimators=750, max_depth=4, learning_rate=0.015, subsample=0.9, colsample_bytree=0.9, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1))]),
            w_bal,
        ))

    rows = []
    for name, model, weight in candidates:
        print("  BCSC:", name)
        model.fit(X_train, y_train, model__sample_weight=weight)
        scores = model.predict_proba(X_valid)[:, 1]
        th = choose_threshold(y_valid, scores, w_valid)
        rows.append({"model": name, "features": len(features), **metrics(y_valid, scores, th, w_valid), **top_decile_lift(y_valid, scores, w_valid)})
    table = pd.DataFrame(rows).sort_values(["roc_auc", "top_decile_lift"], ascending=False)
    table.to_csv(BENCH / "path1_bcsc_interaction_feature_metrics.csv", index=False)
    save_json(table.iloc[0].to_dict(), BENCH / "path1_bcsc_interaction_best_metrics.json")
    return table


def horizon_label(events: pd.Series, months: pd.Series, horizon: float = 60.0) -> pd.Series:
    y = pd.Series(index=events.index, dtype=float)
    y[(events.eq(1)) & (months <= horizon)] = 1.0
    y[(months >= horizon) & ~((events.eq(1)) & (months <= horizon))] = 0.0
    return y


def load_metabric():
    patient = pd.read_csv(METABRIC_DATA_DIR / "data_clinical_patient.txt", sep="\t", comment="#")
    sample = pd.read_csv(METABRIC_DATA_DIR / "data_clinical_sample.txt", sep="\t", comment="#")
    clinical = patient.merge(sample, on="PATIENT_ID", how="inner", suffixes=("_PATIENT", "_SAMPLE")).set_index("SAMPLE_ID", drop=False)
    expr_raw = pd.read_csv(METABRIC_DATA_DIR / "data_mrna_illumina_microarray_zscores_ref_diploid_samples.txt", sep="\t")
    expr_raw = expr_raw.dropna(subset=["Hugo_Symbol"]).copy()
    expr_raw["Hugo_Symbol"] = expr_raw["Hugo_Symbol"].astype(str)
    expr_raw = expr_raw.groupby("Hugo_Symbol", sort=False).mean(numeric_only=True)
    expression = expr_raw.T.astype("float32")
    expression.index.name = "SAMPLE_ID"
    shared = clinical.index.intersection(expression.index)
    return clinical.loc[shared].copy(), expression.loc[shared].copy()


def clinical_matrix(clinical: pd.DataFrame) -> pd.DataFrame:
    numeric = [c for c in ["AGE_AT_DIAGNOSIS", "LYMPH_NODES_EXAMINED_POSITIVE", "NPI", "GRADE", "TUMOR_SIZE", "TUMOR_STAGE", "TMB_NONSYNONYMOUS", "COHORT"] if c in clinical.columns]
    categorical = [c for c in ["CELLULARITY", "CHEMOTHERAPY", "ER_IHC", "ER_STATUS", "HER2_SNP6", "HER2_STATUS", "HORMONE_THERAPY", "INFERRED_MENOPAUSAL_STATE", "PR_STATUS", "RADIO_THERAPY", "HISTOLOGICAL_SUBTYPE", "BREAST_SURGERY", "CANCER_TYPE_DETAILED", "THREEGENE", "INTCLUST"] if c in clinical.columns]
    clin = pd.concat([clinical[numeric].apply(pd.to_numeric, errors="coerce"), pd.get_dummies(clinical[categorical].fillna("Missing").astype(str))], axis=1)
    if {"TUMOR_SIZE", "LYMPH_NODES_EXAMINED_POSITIVE"}.issubset(clinical.columns):
        clin["tumor_size_x_nodes"] = pd.to_numeric(clinical["TUMOR_SIZE"], errors="coerce") * pd.to_numeric(clinical["LYMPH_NODES_EXAMINED_POSITIVE"], errors="coerce")
    if {"GRADE", "TUMOR_STAGE"}.issubset(clinical.columns):
        clin["grade_x_stage"] = pd.to_numeric(clinical["GRADE"], errors="coerce") * pd.to_numeric(clinical["TUMOR_STAGE"], errors="coerce")
    return clin


GENE_SIGNATURES = {
    "proliferation": ["MKI67", "AURKA", "BIRC5", "CCNB1", "MYBL2", "UBE2C", "CDC20", "MELK", "CEP55"],
    "er_signaling": ["ESR1", "PGR", "FOXA1", "GATA3", "BCL2", "XBP1", "TFF1", "GREB1"],
    "her2_signaling": ["ERBB2", "GRB7", "PGAP3", "STARD3", "MIEN1"],
    "basal_tnbc": ["KRT5", "KRT14", "KRT17", "EGFR", "FOXC1", "VIM", "SNAI2"],
    "immune": ["CD8A", "CD8B", "CD3D", "CD3E", "GZMB", "PRF1", "CXCL9", "CXCL10", "PDCD1", "CD274"],
    "invasion_emt": ["VIM", "SNAI1", "SNAI2", "TWIST1", "ZEB1", "MMP9", "MMP2"],
}


def signature_matrix(expression: pd.DataFrame) -> pd.DataFrame:
    sig = pd.DataFrame(index=expression.index)
    for name, genes in GENE_SIGNATURES.items():
        available = [g for g in genes if g in expression.columns]
        if available:
            sig[f"sig_{name}"] = expression[available].mean(axis=1)
            sig[f"sig_{name}_max"] = expression[available].max(axis=1)
    if {"sig_proliferation", "sig_er_signaling"}.issubset(sig.columns):
        sig["sig_prolif_minus_er"] = sig["sig_proliferation"] - sig["sig_er_signaling"]
    if {"sig_her2_signaling", "sig_er_signaling"}.issubset(sig.columns):
        sig["sig_her2_minus_er"] = sig["sig_her2_signaling"] - sig["sig_er_signaling"]
    return sig


def rank_genes(expression: pd.DataFrame, train_idx: pd.Index, y_train: pd.Series) -> list[str]:
    X = SimpleImputer(strategy="median").fit_transform(expression.loc[train_idx])
    scores, _ = f_classif(X, y_train)
    scores = np.nan_to_num(scores, nan=-np.inf, posinf=np.inf, neginf=-np.inf)
    return expression.columns[np.argsort(scores)[::-1]].tolist()


def run_metabric_signature_benchmark() -> pd.DataFrame:
    print("Running METABRIC signature benchmark...")
    clinical, expression = load_metabric()
    labels = pd.DataFrame(index=clinical.index)
    labels["os_months"] = pd.to_numeric(clinical["OS_MONTHS"], errors="coerce")
    labels["os_event"] = clinical["OS_STATUS"].map({"1:DECEASED": 1.0, "0:LIVING": 0.0})
    labels["os_5yr_event"] = horizon_label(labels["os_event"], labels["os_months"], 60)
    y = labels["os_5yr_event"].dropna().astype(int)
    trainval_idx, hold_idx = train_test_split(y.index, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
    train_idx, valid_idx = train_test_split(pd.Index(trainval_idx), test_size=0.25, stratify=y.loc[trainval_idx], random_state=43)
    train_idx, valid_idx, hold_idx = pd.Index(train_idx), pd.Index(valid_idx), pd.Index(hold_idx)
    clin = clinical_matrix(clinical)
    sig = signature_matrix(expression)
    ranked = rank_genes(expression, train_idx, y.loc[train_idx])
    rows = []
    for k in [50, 100, 300]:
        genes = ranked[:k]
        full = pd.concat([clinical_matrix(clinical), signature_matrix(expression), expression[genes]], axis=1)
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        X_train = scaler.fit_transform(imputer.fit_transform(full.loc[train_idx]))
        X_valid = scaler.transform(imputer.transform(full.loc[valid_idx].reindex(columns=full.columns, fill_value=0)))
        X_hold = scaler.transform(imputer.transform(full.loc[hold_idx].reindex(columns=full.columns, fill_value=0)))
        models = []
        if LGBMClassifier is not None:
            models.append(("signature_lightgbm", LGBMClassifier(n_estimators=650, learning_rate=0.02, num_leaves=15, min_child_samples=10, subsample=0.9, colsample_bytree=0.75, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)))
        if XGBClassifier is not None:
            models.append(("signature_xgboost", XGBClassifier(n_estimators=520, max_depth=2, learning_rate=0.02, subsample=0.9, colsample_bytree=0.75, reg_lambda=2.0, scale_pos_weight=(y.loc[train_idx].eq(0).sum() / y.loc[train_idx].eq(1).sum()), eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)))
        for name, model in models:
            print("  METABRIC:", name, "k=", k)
            model.fit(X_train, y.loc[train_idx])
            valid_scores = model.predict_proba(X_valid)[:, 1]
            hold_scores = model.predict_proba(X_hold)[:, 1]
            th = choose_threshold(y.loc[valid_idx], valid_scores)
            rows.append({"model": name, "k_genes": k, "signature_features": sig.shape[1], "validation_auc": roc_auc_score(y.loc[valid_idx], valid_scores), **{f"holdout_{kk}": vv for kk, vv in metrics(y.loc[hold_idx], hold_scores, th).items()}})
    table = pd.DataFrame(rows).sort_values(["holdout_roc_auc", "holdout_balanced_accuracy"], ascending=False)
    table.to_csv(BENCH / "path2_metabric_signature_survival_metrics.csv", index=False)
    save_json(table.iloc[0].to_dict(), BENCH / "path2_metabric_signature_best_metrics.json")
    sig.to_csv(BENCH / "path2_metabric_gene_signature_scores.csv")
    return table


def main() -> None:
    bcsc = run_bcsc_feature_engineering()
    met = run_metabric_signature_benchmark()
    summary = pd.DataFrame([
        {
            "path": "Path 1 - BCSC",
            "benchmark": "Interaction features + balanced weights",
            "best_model": bcsc.iloc[0]["model"],
            "auc": round(float(bcsc.iloc[0]["roc_auc"]), 3),
            "balanced_accuracy": round(float(bcsc.iloc[0]["balanced_accuracy"]), 3),
            "sensitivity": round(float(bcsc.iloc[0]["sensitivity"]), 3),
            "specificity": round(float(bcsc.iloc[0]["specificity"]), 3),
            "top_decile_lift": round(float(bcsc.iloc[0]["top_decile_lift"]), 2),
        },
        {
            "path": "Path 2 - METABRIC",
            "benchmark": "Gene signatures + clinical + selected genes",
            "best_model": met.iloc[0]["model"],
            "auc": round(float(met.iloc[0]["holdout_roc_auc"]), 3),
            "balanced_accuracy": round(float(met.iloc[0]["holdout_balanced_accuracy"]), 3),
            "sensitivity": round(float(met.iloc[0]["holdout_sensitivity"]), 3),
            "specificity": round(float(met.iloc[0]["holdout_specificity"]), 3),
            "top_decile_lift": "",
        },
    ])
    summary.to_csv(BENCH / "feature_engineered_benchmark_summary.csv", index=False)
    print(summary.to_string(index=False))
    print("Outputs:", BENCH)


main()


## Decision-Support High-Value Tasks and Calibration

This section adds scientifically valid high-score decision-support tasks: BCSC calibration, subtype top-2/high-confidence reporting, and receptor-status prediction from gene expression after removing direct receptor-status clinical leakage columns.


In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, StackingClassifier
from sklearn.feature_selection import f_classif
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    top_k_accuracy_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBMClassifier = None

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None


RANDOM_STATE = 42
ROOT = Path.cwd()
BCSC_DATA_DIR = ROOT / "external_datasets" / "BCSC"
METABRIC_DATA_DIR = ROOT / "external_datasets" / "METABRIC" / "brca_metabric_direct"
OUT = ROOT / "outputs" / "clean_two_path_complete"
ENH = OUT / "decision_support_enhancements"
MODELS = ENH / "models"
for path in [ENH, MODELS]:
    path.mkdir(parents=True, exist_ok=True)


def save_json(obj: dict, path: Path) -> None:
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def expected_calibration_error(y_true, prob, sample_weight=None, n_bins=10) -> tuple[float, pd.DataFrame]:
    y_true = np.asarray(y_true)
    prob = np.asarray(prob)
    w = np.ones_like(prob, dtype=float) if sample_weight is None else np.asarray(sample_weight, dtype=float)
    bins = np.linspace(0, 1, n_bins + 1)
    rows = []
    ece = 0.0
    total_w = w.sum()
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (prob >= lo) & (prob <= hi if i == n_bins - 1 else prob < hi)
        if not mask.any():
            rows.append({"bin": i + 1, "lower": lo, "upper": hi, "weight": 0, "mean_predicted": np.nan, "observed_rate": np.nan, "abs_gap": np.nan})
            continue
        bw = w[mask].sum()
        mean_pred = np.average(prob[mask], weights=w[mask])
        obs = np.average(y_true[mask], weights=w[mask])
        gap = abs(mean_pred - obs)
        ece += (bw / total_w) * gap
        rows.append({"bin": i + 1, "lower": lo, "upper": hi, "weight": bw, "mean_predicted": mean_pred, "observed_rate": obs, "abs_gap": gap})
    return float(ece), pd.DataFrame(rows)


def run_bcsc_calibration() -> pd.DataFrame:
    print("Running BCSC calibration benchmark...")
    cols = [
        "menopaus", "agegrp", "density", "race", "hispanic", "bmi",
        "agefirst", "nrelbc", "brstproc", "lastmamm", "surgmeno", "hrt",
        "invasive", "cancer", "training", "count",
    ]
    features = cols[:12]
    bcsc = pd.read_csv(BCSC_DATA_DIR / "bcsc_risk_estimation_dataset_1" / "risk.txt", sep=r"\s+", header=None, names=cols, engine="python")
    for col in features + ["cancer", "training"]:
        bcsc[col] = bcsc[col].astype(str)
    bcsc["count"] = pd.to_numeric(bcsc["count"], errors="coerce").fillna(0).astype(float)

    train_df = bcsc[bcsc["training"].eq("1")].copy()
    valid_df = bcsc[bcsc["training"].eq("0")].copy()
    train_idx, cal_idx = train_test_split(train_df.index, test_size=0.25, stratify=train_df["cancer"].astype(int), random_state=RANDOM_STATE)
    onehot = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), features)])
    base = Pipeline([("prep", onehot), ("model", LogisticRegression(max_iter=10000, C=1.0, random_state=RANDOM_STATE))])
    base.fit(train_df.loc[train_idx, features], train_df.loc[train_idx, "cancer"].astype(int), model__sample_weight=train_df.loc[train_idx, "count"])

    cal_y = train_df.loc[cal_idx, "cancer"].astype(int).to_numpy()
    cal_w = train_df.loc[cal_idx, "count"].to_numpy()
    cal_scores = base.predict_proba(train_df.loc[cal_idx, features])[:, 1]
    valid_y = valid_df["cancer"].astype(int).to_numpy()
    valid_w = valid_df["count"].to_numpy()
    raw_scores = base.predict_proba(valid_df[features])[:, 1]

    platt = LogisticRegression(max_iter=10000)
    platt.fit(cal_scores.reshape(-1, 1), cal_y, sample_weight=cal_w)
    platt_scores = platt.predict_proba(raw_scores.reshape(-1, 1))[:, 1]

    isotonic = IsotonicRegression(out_of_bounds="clip")
    isotonic.fit(cal_scores, cal_y, sample_weight=cal_w)
    iso_scores = isotonic.transform(raw_scores)

    rows = []
    for name, scores in [("raw_logistic", raw_scores), ("platt_scaled", platt_scores), ("isotonic", iso_scores)]:
        ece, cal_table = expected_calibration_error(valid_y, scores, valid_w, n_bins=10)
        cal_table.to_csv(ENH / f"path1_bcsc_calibration_bins_{name}.csv", index=False)
        rows.append({
            "model": name,
            "roc_auc": roc_auc_score(valid_y, scores, sample_weight=valid_w),
            "average_precision": average_precision_score(valid_y, scores, sample_weight=valid_w),
            "brier_score": brier_score_loss(valid_y, scores, sample_weight=valid_w),
            "expected_calibration_error": ece,
            "mean_predicted_risk_percent": np.average(scores, weights=valid_w) * 100,
            "observed_event_rate_percent": np.average(valid_y, weights=valid_w) * 100,
        })
    table = pd.DataFrame(rows).sort_values(["brier_score", "expected_calibration_error"])
    table.to_csv(ENH / "path1_bcsc_calibration_metrics.csv", index=False)
    save_json(table.iloc[0].to_dict(), ENH / "path1_bcsc_best_calibration_metrics.json")
    return table


def horizon_label(events: pd.Series, months: pd.Series, horizon: float = 60.0) -> pd.Series:
    y = pd.Series(index=events.index, dtype=float)
    y[(events.eq(1)) & (months <= horizon)] = 1.0
    y[(months >= horizon) & ~((events.eq(1)) & (months <= horizon))] = 0.0
    return y


def load_metabric():
    patient = pd.read_csv(METABRIC_DATA_DIR / "data_clinical_patient.txt", sep="\t", comment="#")
    sample = pd.read_csv(METABRIC_DATA_DIR / "data_clinical_sample.txt", sep="\t", comment="#")
    clinical = patient.merge(sample, on="PATIENT_ID", how="inner", suffixes=("_PATIENT", "_SAMPLE")).set_index("SAMPLE_ID", drop=False)
    expr_raw = pd.read_csv(METABRIC_DATA_DIR / "data_mrna_illumina_microarray_zscores_ref_diploid_samples.txt", sep="\t")
    expr_raw = expr_raw.dropna(subset=["Hugo_Symbol"]).copy()
    expr_raw["Hugo_Symbol"] = expr_raw["Hugo_Symbol"].astype(str)
    expr_raw = expr_raw.groupby("Hugo_Symbol", sort=False).mean(numeric_only=True)
    expression = expr_raw.T.astype("float32")
    expression.index.name = "SAMPLE_ID"
    shared = clinical.index.intersection(expression.index)
    return clinical.loc[shared].copy(), expression.loc[shared].copy()


def clinical_matrix(clinical: pd.DataFrame) -> pd.DataFrame:
    numeric = [c for c in ["AGE_AT_DIAGNOSIS", "LYMPH_NODES_EXAMINED_POSITIVE", "NPI", "GRADE", "TUMOR_SIZE", "TUMOR_STAGE", "TMB_NONSYNONYMOUS", "COHORT"] if c in clinical.columns]
    categorical = [c for c in ["CELLULARITY", "CHEMOTHERAPY", "ER_IHC", "ER_STATUS", "HER2_SNP6", "HER2_STATUS", "HORMONE_THERAPY", "INFERRED_MENOPAUSAL_STATE", "PR_STATUS", "RADIO_THERAPY", "HISTOLOGICAL_SUBTYPE", "BREAST_SURGERY", "CANCER_TYPE_DETAILED", "THREEGENE", "INTCLUST"] if c in clinical.columns]
    return pd.concat([clinical[numeric].apply(pd.to_numeric, errors="coerce"), pd.get_dummies(clinical[categorical].fillna("Missing").astype(str))], axis=1)


def rank_genes(expression, idx, y):
    X = SimpleImputer(strategy="median").fit_transform(expression.loc[idx])
    scores, _ = f_classif(X, y.loc[idx])
    scores = np.nan_to_num(scores, nan=-np.inf, posinf=np.inf, neginf=-np.inf)
    return expression.columns[np.argsort(scores)[::-1]].tolist()


def build_scaled(train_df, hold_df):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    X_train = scaler.fit_transform(imputer.fit_transform(train_df))
    X_hold = scaler.transform(imputer.transform(hold_df.reindex(columns=train_df.columns, fill_value=0)))
    return X_train, X_hold


def binary_metrics(y_true, scores):
    pred = (scores >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "roc_auc": roc_auc_score(y_true, scores),
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "sensitivity": recall_score(y_true, pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        "precision": precision_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
    }


def run_metabric_decision_support_tasks() -> tuple[pd.DataFrame, pd.DataFrame]:
    print("Running METABRIC high-score decision-support tasks...")
    clinical, expression = load_metabric()
    base_clin = clinical_matrix(clinical)

    # Subtype top-k and high-confidence benchmark.
    subtype_y = clinical["CLAUDIN_SUBTYPE"].replace({"NC": np.nan}).dropna().astype(str)
    subtype_idx = subtype_y.index
    train_idx, hold_idx = train_test_split(subtype_idx, test_size=0.2, stratify=subtype_y, random_state=RANDOM_STATE)
    train_idx, hold_idx = pd.Index(train_idx), pd.Index(hold_idx)
    ranked = rank_genes(expression, train_idx, subtype_y)
    selected = ranked[:500]
    subtype_train = pd.concat([expression.loc[train_idx, selected], base_clin.loc[train_idx]], axis=1)
    subtype_hold = pd.concat([expression.loc[hold_idx, selected], base_clin.loc[hold_idx]], axis=1)
    X_train, X_hold = build_scaled(subtype_train, subtype_hold)
    enc = LabelEncoder()
    y_train = enc.fit_transform(subtype_y.loc[train_idx])
    y_hold = enc.transform(subtype_y.loc[hold_idx])
    estimators = [
        ("logreg", LogisticRegression(max_iter=10000, C=0.1, class_weight="balanced", random_state=RANDOM_STATE)),
        ("extra", ExtraTreesClassifier(n_estimators=600, min_samples_leaf=2, max_features="sqrt", class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
    ]
    if LGBMClassifier is not None:
        estimators.append(("lgbm", LGBMClassifier(n_estimators=450, learning_rate=0.025, num_leaves=15, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)))
    stack = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(max_iter=10000, C=0.5, class_weight="balanced"), stack_method="predict_proba", cv=5, n_jobs=-1)
    stack.fit(X_train, y_train)
    proba = stack.predict_proba(X_hold)
    pred = proba.argmax(axis=1)
    confidence = proba.max(axis=1)
    subtype_rows = [{
        "task": "CLAUDIN subtype 6-class stacking",
        "balanced_accuracy": balanced_accuracy_score(y_hold, pred),
        "accuracy": accuracy_score(y_hold, pred),
        "top2_accuracy": top_k_accuracy_score(y_hold, proba, k=2, labels=np.arange(len(enc.classes_))),
        "high_confidence_threshold": 0.70,
        "high_confidence_coverage": float((confidence >= 0.70).mean()),
        "high_confidence_accuracy": float(accuracy_score(y_hold[confidence >= 0.70], pred[confidence >= 0.70])) if (confidence >= 0.70).any() else np.nan,
        "classes": ",".join(enc.classes_),
    }]
    subtype_table = pd.DataFrame(subtype_rows)
    subtype_table.to_csv(ENH / "path2_subtype_topk_high_confidence_metrics.csv", index=False)

    # Easier but clinically useful receptor tasks.
    receptor_rows = []
    receptor_targets = {
        "ER_STATUS": {"Positive": 1, "Negative": 0},
        "PR_STATUS": {"Positive": 1, "Negative": 0},
        "HER2_STATUS": {"Positive": 1, "Negative": 0},
    }
    receptor_leakage_cols = [
        "ER_STATUS", "ER_IHC", "PR_STATUS", "HER2_STATUS", "HER2_SNP6",
        "THREEGENE",
    ]
    receptor_clin = base_clin.drop(columns=[c for c in base_clin.columns if any(token in c for token in receptor_leakage_cols)], errors="ignore")
    for target, mapping in receptor_targets.items():
        y = clinical[target].map(mapping).dropna().astype(int)
        if y.nunique() < 2:
            continue
        idx = y.index
        train_idx, hold_idx = train_test_split(idx, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
        train_idx, hold_idx = pd.Index(train_idx), pd.Index(hold_idx)
        ranked = rank_genes(expression, train_idx, y)
        genes = ranked[:300]
        train_df = pd.concat([expression.loc[train_idx, genes], receptor_clin.loc[train_idx]], axis=1)
        hold_df = pd.concat([expression.loc[hold_idx, genes], receptor_clin.loc[hold_idx]], axis=1)
        X_train, X_hold = build_scaled(train_df, hold_df)
        if LGBMClassifier is not None:
            model = LGBMClassifier(n_estimators=450, learning_rate=0.025, num_leaves=15, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
        else:
            model = LogisticRegression(max_iter=10000, C=0.1, class_weight="balanced", random_state=RANDOM_STATE)
        model.fit(X_train, y.loc[train_idx])
        scores = model.predict_proba(X_hold)[:, 1]
        receptor_rows.append({"task": f"{target} prediction", "samples": int(len(y)), "positive_rate": float(y.mean()), **binary_metrics(y.loc[hold_idx].values, scores)})
    receptor_table = pd.DataFrame(receptor_rows).sort_values("roc_auc", ascending=False)
    receptor_table.to_csv(ENH / "path2_receptor_prediction_metrics.csv", index=False)
    return subtype_table, receptor_table


def main() -> None:
    bcsc = run_bcsc_calibration()
    subtype, receptor = run_metabric_decision_support_tasks()
    summary = []
    best_cal = bcsc.iloc[0]
    summary.append({
        "section": "BCSC calibration",
        "task": best_cal["model"],
        "primary_metric": "Brier score",
        "result": round(float(best_cal["brier_score"]), 5),
        "secondary_metric": "ECE",
        "secondary_result": round(float(best_cal["expected_calibration_error"]), 5),
    })
    s = subtype.iloc[0]
    summary.append({
        "section": "METABRIC subtype",
        "task": s["task"],
        "primary_metric": "Top-2 accuracy",
        "result": round(float(s["top2_accuracy"]), 3),
        "secondary_metric": "High-confidence accuracy",
        "secondary_result": round(float(s["high_confidence_accuracy"]), 3),
    })
    for _, row in receptor.iterrows():
        summary.append({
            "section": "METABRIC receptor",
            "task": row["task"],
            "primary_metric": "ROC-AUC",
            "result": round(float(row["roc_auc"]), 3),
            "secondary_metric": "Accuracy",
            "secondary_result": round(float(row["accuracy"]), 3),
        })
    summary_df = pd.DataFrame(summary)
    summary_df.to_csv(ENH / "decision_support_enhancement_summary.csv", index=False)
    print(summary_df.to_string(index=False))
    print("Outputs:", ENH)


main()
